In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from scipy import interpolate
import math
from scipy.ndimage import gaussian_filter
import MDAnalysis as mda
from MDAnalysis.analysis import align
import matplotlib.colors as mcolors
from matplotlib.cm import get_cmap
from matplotlib.colors import TwoSlopeNorm, ListedColormap

In [ ]:
###################################################################
# Plot options
###################################################################
font = {'family' : 'sans-serif',
        'serif'   : 'Arial',
        #'sans-serif'    : 'Computer Modern Sans serif',
        'style'   : 'normal',
        'variant'   : 'normal',
        'stretch'   : 'normal',
        'weight'   : 'normal',
        'size'   : 30}
plt.rc('font', **font)
#plt.rc('text', usetex=True)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (7,5) #(3.37,2.25)

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:

# Ignore the initial lines in each file
ignore = 0

# Initialize lists to store data for each COLVAR

def read_xvg_numeric(path):
    numeric_lines = []
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            # keep only lines whose first non-space char looks numeric: digit, -, or .
            c = s[0]
            if c.isdigit() or c in "-.":
                numeric_lines.append(s)

    # Now parse the clean numeric block
    return np.loadtxt(numeric_lines)

time_list = []
cmap_list = []

# Loop through the 20 COLVAR files
for colvar_num in range(8):
    file_path = f"/project/zerze/krahimin/proteinA_all/2B/round3/data/mindist/2000ns_mindist_nd{colvar_num}.xvg"
    colvar_data = read_xvg_numeric(file_path)  # Read data ignoring initial lines
   # print(colvar_num,len(colvar_data))
    time_list.append(colvar_data[::50, 0])  # Extract time (column 0)
    cmap_list.append(colvar_data[::50, 1])  # Extract cmap (column 2)

# Set up the figure with 5 rows and 4 columns
fig, axs = plt.subplots(4, 2, figsize=(15, 15), sharex=True, sharey=True)

# Flatten the axes for easy iteration
axs = axs.flatten()

# Plot each COLVAR on its own subplot
for i in range(8):
    axs[i].plot(time_list[i] / 1000, cmap_list[i], 'o', markersize=1, label=f'COLVAR {i}')
 #   axs[i].legend()  # Add legend for each plot
    axs[i].set_title(f'COLVAR {i}',fontsize=20)  # Optional: Add a title for each subplot

# Add labels to the shared axes
for ax in axs[-4:]:  # Only the bottom row
    ax.set_xticks(np.arange(0, 2001, 1000))
    ax.set_xlabel('Time (ns)')
for ax in axs[::4]:  # Only the first column
    ax.set_yticks(np.arange(0, 6, 1))
    ax.set_ylabel('Mindist (nm)')

# Adjust layout for better spacing
plt.tight_layout()

plt.show()


In [ ]:

def read_xvg_numeric(path):
    numeric_lines = []
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            if s[0].isdigit() or s[0] in "-.":
                numeric_lines.append(s)
    return np.loadtxt(numeric_lines)



all_mindist_eq = []

for nd in range(8):
    path = f"/project/zerze/krahimin/proteinA_all/2B/round3/data/mindist/2000ns_mindist_nd{nd}.xvg"
    data = read_xvg_numeric(path)

    time_ps = data[:, 0]
    mindist = data[:, 1]

    all_mindist_eq.append(mindist)

all_mindist_eq = np.concatenate(all_mindist_eq)
plt.figure(figsize=(8, 6))

plt.hist(
    all_mindist_eq,
    bins=100,
    density=True,
    cumulative=True,
    histtype="step",
    linewidth=2
)

plt.xlabel("Minimum distance (nm)")
plt.ylabel("Cumulative probability")
plt.yticks(np.arange(0,1, 0.1))
plt.xticks(np.arange(0,6,1))
plt.tight_layout()
plt.show()



In [ ]:

# Ignore the initial lines in each file
ignore = 0

# Initialize lists to store data for each COLVAR
time_list = []
cmap_list = []

# Loop through the 20 COLVAR files
for colvar_num in range(16):
    file_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{colvar_num}/everynth"
    colvar_data = np.genfromtxt(file_path)  # Read data ignoring initial lines
   # print(colvar_num,len(colvar_data))
    time_list.append(colvar_data[:, 0])  # Extract time (column 0)
    cmap_list.append(colvar_data[:, 2])  # Extract cmap (column 2)

# Set up the figure with 5 rows and 4 columns
fig, axs = plt.subplots(4, 4, figsize=(15, 15), sharex=True, sharey=True)

# Flatten the axes for easy iteration
axs = axs.flatten()

# Plot each COLVAR on its own subplot
for i in range(16):
    axs[i].plot(time_list[i] / 1000, cmap_list[i], 'o', markersize=1, label=f'COLVAR {i}')
 #   axs[i].legend()  # Add legend for each plot
    axs[i].set_title(f'COLVAR {i}',fontsize=20)  # Optional: Add a title for each subplot

# Add labels to the shared axes
for ax in axs[-4:]:  # Only the bottom row
    ax.set_xticks(np.arange(0, 2001, 1000))
    ax.set_xlabel('Time (ns)')
for ax in axs[::4]:  # Only the first column
    ax.set_yticks(np.arange(0, 1, 0.3))
    ax.set_ylabel('Q')

# Adjust layout for better spacing
plt.tight_layout()
plt.savefig("2B_linker_colvars_Q_time.png", dpi=400,  bbox_inches='tight')

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -------- settings --------
ignore = 0  # lines to skip at the top of each file, if needed
n_walkers = 16
base = "/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/"
fname = "everynth"  # same file name for all walkers
# --------------------------

# Build paths colvar0..colvar16
paths = [f"{base}/{i}/{fname}" for i in range(n_walkers)]

# Load data for each walker
COLVARS = [np.genfromtxt(p)[ignore:, :] for p in paths]

# Use time from walker 0 (assumes aligned sampling across walkers)
time = np.copy(COLVARS[0][:, 0])
# Extract the 3rd column (index 2) as the cmap value for each walker
cmap_values = np.array([cv[:, 2] for cv in COLVARS])

# Average across walkers at each time step
cmap_avg = np.mean(cmap_values, axis=0)

# Convert time to ns
time_ns = time / 1000.0

# 50 ns binning of the average
bin_size = 50.0
bins = np.arange(0, max(time_ns) + bin_size, bin_size)
bin_indices = np.digitize(time_ns, bins)
bin_means = [np.mean(cmap_avg[bin_indices == i]) for i in range(1, len(bins))]
bin_centers = bins[:-1] + bin_size / 2.0

# ----- plotting -----
plt.figure(figsize=(10, 12))
colors = plt.cm.viridis(np.linspace(0, 1, n_walkers))

# Plot all walkers
for i in range(n_walkers):
    plt.scatter(time_ns, cmap_values[i], color=colors[i], s=1, alpha=0.5, label=f"Walker {i}")

# Average across walkers
plt.plot(time_ns, cmap_avg, color='black', linewidth=2, label=f"Average across {n_walkers} walkers")

# 50 ns binned average as red points (hidden from auto-legend)
plt.scatter(bin_centers, bin_means, color='red', zorder=2, s=20, label="_nolegend_")

# Labels and guide line
plt.xlabel("time (ns)", fontsize=30)
plt.ylabel("Q", fontsize=30)
plt.xticks( np.arange(0,2001,1000),fontsize=30)
plt.yticks( fontsize=30)
plt.axvline(500, color='red', linestyle='dashed', label='Equilibration')

# Legend (add a small marker entry for the 50 ns average)
handles, labels = plt.gca().get_legend_handles_labels()
custom_handle = Line2D([0], [0], marker='o', color='red', linestyle='None', markersize=4, label="50 ns Average")
handles.append(custom_handle)
labels.append("50 ns Average")

plt.legend(
    handles=handles,
    labels=labels,
    fontsize=15,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.25),
    scatterpoints=3,
    markerscale=4,
    ncol=4
)

plt.tight_layout()
plt.savefig("2B_linker_Q_time.png", dpi=400,  bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -------- settings --------
ignore = 0  # lines to skip at the top of each file, if needed
n_walkers = 16
base = "/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/"
fname = "pasted_2000ns"  # same file name for all walkers
# --------------------------

# Build paths colvar0..colvar16
paths = [f"{base}/{i}/{fname}" for i in range(n_walkers)]

# Load data for each walker
COLVARS = [np.genfromtxt(p)[ignore:, :] for p in paths]

# Use time from walker 0 (assumes aligned sampling across walkers)
time = np.copy(COLVARS[0][:, 0])
# Extract the 3rd column (index 2) as the cmap value for each walker
cmap_values = np.array([cv[:, 7] for cv in COLVARS])

# Average across walkers at each time step
cmap_avg = np.mean(cmap_values, axis=0)

# Convert time to ns
time_ns = time / 1000.0

# 50 ns binning of the average
bin_size = 50.0
bins = np.arange(0, max(time_ns) + bin_size, bin_size)
bin_indices = np.digitize(time_ns, bins)
bin_means = [np.mean(cmap_avg[bin_indices == i]) for i in range(1, len(bins))]
bin_centers = bins[:-1] + bin_size / 2.0

# ----- plotting -----
plt.figure(figsize=(10, 12))
colors = plt.cm.viridis(np.linspace(0, 1, n_walkers))

# Plot all walkers
for i in range(n_walkers):
    plt.scatter(time_ns, cmap_values[i], color=colors[i], s=1, alpha=0.5, label=f"Walker {i}")

# Average across walkers
plt.plot(time_ns, cmap_avg, color='black', linewidth=2, label=f"Average across {n_walkers} walkers")

# 50 ns binned average as red points (hidden from auto-legend)
plt.scatter(bin_centers, bin_means, color='red', zorder=2, s=20, label="_nolegend_")

# Labels and guide line
plt.xlabel("time (ns)", fontsize=30)
plt.ylabel("Rg (nm)", fontsize=30)
plt.xticks( np.arange(0,2001,1000),fontsize=30)
plt.yticks( fontsize=30)
plt.axvline(500, color='red', linestyle='dashed', label='Equilibration')

# Legend (add a small marker entry for the 50 ns average)
handles, labels = plt.gca().get_legend_handles_labels()
custom_handle = Line2D([0], [0], marker='o', color='red', linestyle='None', markersize=4, label="50 ns Average")
handles.append(custom_handle)
labels.append("50 ns Average")

plt.legend(
    handles=handles,
    labels=labels,
    fontsize=15,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.25),
    scatterpoints=3,
    markerscale=4,
    ncol=4
)

plt.tight_layout()
#plt.savefig("RRM_CHARMM_Q_time.png", dpi=400,  bbox_inches='tight')
plt.show()


In [ ]:
# Taking data from round 1
#ignore=25000 # Ignore this many lines from the beginning of files 500 ns
ignore=0
COLVAR1=np.genfromtxt("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/0/everynth")[ignore:,:]

for i in range(1, 16):  # Load data for domains 1 to 24
    file_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/everynth"
    COLVAR1 = np.concatenate((COLVAR1, np.genfromtxt(file_path)[ignore:, :]))
 #   print(len(COLVAR1))

COLVAR=COLVAR1
time=np.copy(COLVAR[:,0])
cmap=np.copy(COLVAR[:,2])
energy=np.copy(COLVAR[:,1])
bias=np.copy(COLVAR[:,5])
#rg= np.copy(COLVAR[:,7])


times=time.reshape((16,int(len(COLVAR)/16))) #divided the time into 8 pieces
energies=energy.reshape((16,int(len(COLVAR)/16)))
biases=bias.reshape((16,int(len(COLVAR)/16)) )
cmaps=cmap.reshape((16,int(len(COLVAR)/16))) 
all_delta_G=[]
for ignore2 in range (1,int(len(COLVAR)/16),1000):
 
    icmap = cmaps[0,:ignore2]
    ienergy = energies[0,:ignore2]
    ibias = biases[0,:ignore2]
    
    for k in range(1,16):
#        time.append(times[k,:ignore2])
        icmap = np.concatenate((icmap,cmaps[k,:ignore2]))
        ienergy = np.concatenate((ienergy,energies[k,:ignore2]))
        ibias = np.concatenate((ibias,biases[k,:ignore2]))
    
                    
    kb=0.00831446 # kJ/mol/K
    temperature=390 # K
    beta=1./(kb*temperature)
    newTemperature=300 # K
    newBeta=1./(kb*newTemperature)
    logweights=beta*ibias+(beta-newBeta)*ienergy
    logweights -= np.amax(logweights)
    weights = np.exp(logweights)
    
    hist, bin_edges = np.histogram(icmap, weights=weights, density=True,bins=40)
    freeEnergy = -(1./beta)*np.log(hist)
    freeEnergy -= np.amin(freeEnergy)
    bin_centers=(bin_edges[:-1]+bin_edges[1:])/2
    y = bin_centers
    
    U_FE_sum = 0
    F_FE_sum = 0
    U_n =0
    F_n =0
    for j in range (40):
            if (y[j] > 0.5):
                U_FE_sum += np.exp(-freeEnergy [j]/(kb*newTemperature))
                U_n +=1
#                    print (x[j][i], y[j][i], H1[i][j])

            elif (y[j] < 0.5):
                F_FE_sum+= np.exp(-freeEnergy [j]/(kb*newTemperature))
                F_n +=1
    delta_G = -(kb*newTemperature)*np.log((F_FE_sum)/(U_FE_sum))
    all_delta_G.append(delta_G)



mkt = np.average(all_delta_G[-1])-kb*newTemperature # minus kt
pkt = np.average(all_delta_G[-1])+kb*newTemperature # plus kt

fig, ax = plt.subplots()
ax.set_ylim(all_delta_G[-1]-10,all_delta_G[-1]+10)

ax.plot(times[0,1::1000]/1000,all_delta_G)
a=ax
a.axhline(y=mkt, color="red", linestyle="--")
a.axhline(y=pkt, color="black", linestyle="--")
a.set_xlabel("time (ns)",fontsize=20)
a.set_ylabel("∆G (kJ/mol)",fontsize=20)
a.tick_params(axis='x', labelsize=20)
a.tick_params(axis='y', labelsize=20)
plt.tight_layout()
plt.savefig("2B_linker_deltaG_0.5_time.png", dpi=400,  bbox_inches='tight')
plt.show()


In [ ]:
# Taking data from round 1
ignore=25000 # Ignore this many lines from the beginning of files 200 ns
COLVAR1=np.genfromtxt("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/0/everynth")[ignore:,:]
print(len(COLVAR1))
for i in range(1, 16):  # Load data for domains 1 to 24
    file_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/everynth"
    COLVAR1 = np.concatenate((COLVAR1, np.genfromtxt(file_path)[ignore:, :]))
   # print(len(COLVAR1))

COLVAR=COLVAR1
time=np.copy(COLVAR[:,0])
cmap=np.copy(COLVAR[:,2])
energy=np.copy(COLVAR[:,1])
bias=np.copy(COLVAR[:,5])
lentime=int(len(time)/16)

temperature=390 # K

kb=0.00831446 # kJ/mol/K

beta=1./(kb*temperature)

temps=np.linspace(300,480,30)

neff=np.zeros(temps.shape[0])

for i in range(temps.shape[0]):

    newTemperature=temps[i]

    newBeta=1./(kb*newTemperature)

    #calculate weights at a given temperature (wrt to the simulation temperature)

    logweights=beta*bias+(beta-newBeta)*energy #why does log weights only consider energy? Just because energy is the thing that determines T?2
    logweights -= np.amax(logweights)

    weights=np.exp(logweights)

    neff[i]=(np.power(np.sum(weights),2)/np.sum(np.power(weights,2)))

print (neff, logweights.shape)
plt.plot(temps,neff)
plt.xlabel("Temperature", fontsize=20)

plt.ylabel("neff",fontsize=20)

plt.xticks(np.arange(min(temps), max(temps)+30, 30))

plt.tick_params(axis='x', labelsize=20)

plt.tick_params(axis='y', labelsize=20)

plt.show()

In [ ]:
# Taking data from round 1
ignore=50000 # Ignore this many lines from the beginning of files 200 ns
COLVAR1=np.genfromtxt("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/0/everynth")[ignore:,:]
print(len(COLVAR1))
for i in range(1, 16):  # Load data for domains 1 to 24
    file_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/everynth"
    COLVAR1 = np.concatenate((COLVAR1, np.genfromtxt(file_path)[ignore:, :]))
   # print(len(COLVAR1))

COLVAR=COLVAR1
time=np.copy(COLVAR[:,0])
cmap=np.copy(COLVAR[:,2])
energy=np.copy(COLVAR[:,1])
bias=np.copy(COLVAR[:,5])
lentime=int(len(time)/16)
print(time[0], time[-1])
    
kb=0.00831446 # kJ/mol/K
temperature=390 # K
beta=1./(kb*temperature)
newTemperature=330 # K
newBeta=1./(kb*newTemperature)
logweights=beta*bias+(beta-newBeta)*energy
logweights -= np.amax(logweights)
weights = np.exp(logweights)
hist, bin_edges = np.histogram(cmap, weights=np.exp(logweights), density=True,bins=30)
#freeEnergy = -(1./newBeta)*np.log(hist)/numMolecules
freeEnergy = -(1./newBeta)*np.log(hist)
freeEnergy -= np.amin(freeEnergy)
bin_centers=(bin_edges[:-1]+bin_edges[1:])/2
plt.figure(figsize=(10, 8))
plt.plot(bin_centers,freeEnergy,linewidth=4)
plt.xlabel("Q")
#plt.xticks(fontsize=25)
plt.ylabel("Free energy (kJ/mol)")
plt.ylim(0,41)
plt.yticks(np.arange(0,42, 10))
plt.xticks(np.arange(0,1.01,0.25))
plt.tight_layout()
plt.savefig("2B_linker_FE_Q-1000ns.png", dpi=400)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ignore = 25000  # Ignore this many lines from the beginning

# --- Data Loading Section ---
# We initialize lists to hold the arrays, which is often faster/cleaner 
# than concatenating inside the loop, then stack them at the end.

energy_list = []
bias_list = []
q_domain2_list = []

print("Loading data...")

for i in range(16):  # Loops 0 to 15
    # Define Paths
    pasted_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/pasted_2000ns"
    colvar_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/colvar_dmn2"
    
    # Load Data (Time, Energy, Bias) from pasted_2000ns
    # We only need specific columns to save memory, but slicing after load works too
    data_pasted = np.genfromtxt(pasted_path)[ignore:, :]
    
    # Load Q_domain2 from colvar_dmn2
    # The user specified the "second column", which is index 1 in Python
    data_q = np.genfromtxt(colvar_path)[ignore:, 1]
    
    # Safety Check: Ensure the files have matching lengths after ignore
    if len(data_pasted) != len(data_q):
        print(f"Warning: Length mismatch in replica {i}. Pasted: {len(data_pasted)}, Q: {len(data_q)}")
        # Truncate to the shorter length to avoid crashes
        min_len = min(len(data_pasted), len(data_q))
        data_pasted = data_pasted[:min_len]
        data_q = data_q[:min_len]

    # extract columns: Energy is col 1, Bias is col 5
    energy_list.append(data_pasted[:, 1])
    bias_list.append(data_pasted[:, 5])
    q_domain2_list.append(data_q)

# Concatenate all replicas into single arrays
energy = np.concatenate(energy_list)
bias = np.concatenate(bias_list)
q_domain2 = np.concatenate(q_domain2_list)

print(f"Total frames analyzed: {len(energy)}")

# --- Reweighting Constants ---
kb = 0.00831446  # kJ/mol/K
temperature = 390  # K
beta = 1. / (kb * temperature)
newTemperature = 300  # K
newBeta = 1. / (kb * newTemperature)

# --- Calculate Weights ---
# Weight = exp(beta*V + (beta - beta')*E)
logweights = beta * bias + (beta - newBeta) * energy
logweights -= np.amax(logweights)  # Shift for numerical stability to avoid overflow
weights = np.exp(logweights)

# --- Compute Free Energy ---
# Histogram the new variable (q_domain2)
# Adjusted bins to 50 for potentially finer resolution, revert to 30 if data is sparse
hist, bin_edges = np.histogram(q_domain2, weights=weights, density=True, bins=30)

# Calculate F = -kT * ln(P)
# Added a small epsilon (1e-10) to avoid log(0) errors if a bin is empty
freeEnergy = -(1. / newBeta) * np.log(hist + 1e-10)
freeEnergy -= np.amin(freeEnergy)  # Shift global minimum to 0

bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# --- Plotting ---
plt.figure(figsize=(10, 8))
plt.plot(bin_centers, freeEnergy, linewidth=4, color='blue') # Added color for clarity

plt.xlabel("Q (Domain 2)", fontsize=30)
plt.ylabel("Free energy (kJ/mol)", fontsize=30)

plt.ylim(0, 41)
plt.yticks(np.arange(0, 42, 10), fontsize=30)

# Check your Q range; standard Q is 0 to 1. 
# If Q_domain2 is unnormalized (e.g., number of contacts), you may need to adjust these ticks.
plt.xticks(np.arange(0, 1.01, 0.25), fontsize=30)

plt.tight_layout()
# plt.savefig("FreeEnergy_Q_Domain2.png", dpi=400)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ignore = 25000  # Ignore this many lines from the beginning

# --- Data Loading Section ---
# We initialize lists to hold the arrays, which is often faster/cleaner 
# than concatenating inside the loop, then stack them at the end.

energy_list = []
bias_list = []
q_domain2_list = []

print("Loading data...")

for i in range(16):  # Loops 0 to 15
    # Define Paths
    pasted_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/pasted_2000ns"
    colvar_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/colvar_dmn1"
    
    # Load Data (Time, Energy, Bias) from pasted_2000ns
    # We only need specific columns to save memory, but slicing after load works too
    data_pasted = np.genfromtxt(pasted_path)[ignore:, :]
    
    # Load Q_domain2 from colvar_dmn2
    # The user specified the "second column", which is index 1 in Python
    data_q = np.genfromtxt(colvar_path)[ignore:, 1]
    
    # Safety Check: Ensure the files have matching lengths after ignore
    if len(data_pasted) != len(data_q):
        print(f"Warning: Length mismatch in replica {i}. Pasted: {len(data_pasted)}, Q: {len(data_q)}")
        # Truncate to the shorter length to avoid crashes
        min_len = min(len(data_pasted), len(data_q))
        data_pasted = data_pasted[:min_len]
        data_q = data_q[:min_len]

    # extract columns: Energy is col 1, Bias is col 5
    energy_list.append(data_pasted[:, 1])
    bias_list.append(data_pasted[:, 5])
    q_domain2_list.append(data_q)

# Concatenate all replicas into single arrays
energy = np.concatenate(energy_list)
bias = np.concatenate(bias_list)
q_domain2 = np.concatenate(q_domain2_list)

print(f"Total frames analyzed: {len(energy)}")

# --- Reweighting Constants ---
kb = 0.00831446  # kJ/mol/K
temperature = 390  # K
beta = 1. / (kb * temperature)
newTemperature = 300  # K
newBeta = 1. / (kb * newTemperature)

# --- Calculate Weights ---
# Weight = exp(beta*V + (beta - beta')*E)
logweights = beta * bias + (beta - newBeta) * energy
logweights -= np.amax(logweights)  # Shift for numerical stability to avoid overflow
weights = np.exp(logweights)

# --- Compute Free Energy ---
# Histogram the new variable (q_domain2)
# Adjusted bins to 50 for potentially finer resolution, revert to 30 if data is sparse
hist, bin_edges = np.histogram(q_domain2, weights=weights, density=True, bins=30)

# Calculate F = -kT * ln(P)
# Added a small epsilon (1e-10) to avoid log(0) errors if a bin is empty
freeEnergy = -(1. / newBeta) * np.log(hist + 1e-10)
freeEnergy -= np.amin(freeEnergy)  # Shift global minimum to 0

bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# --- Plotting ---
plt.figure(figsize=(10, 8))
plt.plot(bin_centers, freeEnergy, linewidth=4, color='blue') # Added color for clarity

plt.xlabel("Q (Domain 1)", fontsize=30)
plt.ylabel("Free energy (kJ/mol)", fontsize=30)

plt.ylim(0, 41)
plt.yticks(np.arange(0, 42, 10), fontsize=30)

# Check your Q range; standard Q is 0 to 1. 
# If Q_domain2 is unnormalized (e.g., number of contacts), you may need to adjust these ticks.
plt.xticks(np.arange(0, 1.01, 0.25), fontsize=30)

plt.tight_layout()
# plt.savefig("FreeEnergy_Q_Domain2.png", dpi=400)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ignore = 25000  # Ignore this many lines from the beginning

# --- Data Loading Section ---
energy_list = []
bias_list = []
q1_list = [] # List for Domain 1
q2_list = [] # List for Domain 2

print("Loading data...")

for i in range(16):  # Loops 0 to 15
    # Define Paths
    pasted_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/pasted_2000ns"
    colvar1_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/colvar_dmn1"
    colvar2_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/colvar_dmn2"
    
    # Load Data
    # pasted: col 1=Energy, col 5=Bias
    data_pasted = np.genfromtxt(pasted_path)[ignore:, :]
    
    # Load Q1 (Domain 1) - col 1
    data_q1 = np.genfromtxt(colvar1_path)[ignore:, 1]

    # Load Q2 (Domain 2) - col 1
    data_q2 = np.genfromtxt(colvar2_path)[ignore:, 1]
    
    # --- Synchronization Safety Check ---
    # We must ensure all three arrays (Energy, Q1, Q2) have the exact same length
    len_p = len(data_pasted)
    len_q1 = len(data_q1)
    len_q2 = len(data_q2)

    if not (len_p == len_q1 == len_q2):
        print(f"Warning: Length mismatch in replica {i}. Pasted: {len_p}, Q1: {len_q1}, Q2: {len_q2}")
        # Truncate all to the shortest length found
        min_len = min(len_p, len_q1, len_q2)
        data_pasted = data_pasted[:min_len]
        data_q1 = data_q1[:min_len]
        data_q2 = data_q2[:min_len]

    # Append to lists
    energy_list.append(data_pasted[:, 1])
    bias_list.append(data_pasted[:, 5])
    q1_list.append(data_q1)
    q2_list.append(data_q2)

# Concatenate all replicas
energy = np.concatenate(energy_list)
bias = np.concatenate(bias_list)
q1 = np.concatenate(q1_list)
q2 = np.concatenate(q2_list)

print(f"Total frames analyzed: {len(energy)}")

# --- Reweighting Constants ---
kb = 0.00831446 
temperature = 390 
beta = 1. / (kb * temperature)
newTemperature = 300 
newBeta = 1. / (kb * newTemperature)

# --- Calculate Weights (Shared) ---
logweights = beta * bias + (beta - newBeta) * energy
logweights -= np.amax(logweights)
weights = np.exp(logweights)

# --- Compute Free Energy for Domain 1 ---
hist1, bin_edges1 = np.histogram(q1, weights=weights, density=True, bins=30)
freeEnergy1 = -(1. / newBeta) * np.log(hist1 + 1e-10)
freeEnergy1 -= np.amin(freeEnergy1) # Shift min to 0
bin_centers1 = (bin_edges1[:-1] + bin_edges1[1:]) / 2

# --- Compute Free Energy for Domain 2 ---
hist2, bin_edges2 = np.histogram(q2, weights=weights, density=True, bins=30)
freeEnergy2 = -(1. / newBeta) * np.log(hist2 + 1e-10)
freeEnergy2 -= np.amin(freeEnergy2) # Shift min to 0
bin_centers2 = (bin_edges2[:-1] + bin_edges2[1:]) / 2

# --- Plotting ---
plt.figure(figsize=(10, 8))

# Plot Domain 1 (e.g., Black)
plt.plot(bin_centers1, freeEnergy1, linewidth=4, color='black', label='Domain 1')

# Plot Domain 2 (e.g., Red)
plt.plot(bin_centers2, freeEnergy2, linewidth=4, color='red', label='Domain 2')

plt.xlabel("Q", fontsize=30)
plt.ylabel("Free energy (kJ/mol)", fontsize=30)

plt.ylim(0, 41)
plt.yticks(np.arange(0, 42, 10), fontsize=30)
plt.xticks(np.arange(0, 1.01, 0.25), fontsize=30)

plt.legend(fontsize=20, frameon=False) # Add legend to distinguish lines
plt.tight_layout()

# plt.savefig("FreeEnergy_Comparison.png", dpi=400)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# --- Data Loading Section ---
# (Keeping your original loading logic intact)
ignore = 25000  # Ignore this many lines
# Initial load
COLVAR1 = np.genfromtxt("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/0/pasted_2000ns")[ignore:, :]
print(f"Replica 0 loaded: {len(COLVAR1)} frames")

for i in range(1, 16):  # Load data for replicas 1 to 15
    file_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/pasted_2000ns"
    # Concatenate subsequent files
    COLVAR1 = np.concatenate((COLVAR1, np.genfromtxt(file_path)[ignore:, :]))
    # print(f"Replica {i} loaded")

COLVAR = COLVAR1
time = np.copy(COLVAR[:, 0])
energy = np.copy(COLVAR[:, 1]) # Potential Energy
cmap = np.copy(COLVAR[:, 2])   # Collective Variable (Q)
bias = np.copy(COLVAR[:, 5])   # Bias Potential

print(f"Total frames: {len(time)}")
print(f"Time range: {time[0]} to {time[-1]}")

# --- Reweighting & Plotting Section ---

# Constants
kb = 0.00831446  # kJ/mol/K
original_temp = 390  # K (Simulation Temperature)
beta = 1. / (kb * original_temp)

# Define Target Temperatures: 300 to 480 K with frequency of 10
# arange is [start, stop), so we use 490 to include 480
target_temps = np.arange(300, 490, 10) 

# Setup Plot
fig, ax = plt.subplots(figsize=(12, 10))

# Setup Colormap (Jet goes from Blue/Cold to Red/Hot)
norm = mcolors.Normalize(vmin=target_temps.min(), vmax=target_temps.max())
colormap = cm.jet 
scalar_mappable = cm.ScalarMappable(norm=norm, cmap=colormap)
scalar_mappable.set_array([]) # Necessary for the colorbar

print("Starting reweighting loop...")

for newTemperature in target_temps:
    newBeta = 1. / (kb * newTemperature)
    
    # 1. Calculate Log Weights
    # Formula: Remove bias (beta*bias) and reweight Temp ((beta - newBeta)*energy)
    logweights = beta * bias + (beta - newBeta) * energy
    
    # Shift for numerical stability (prevent overflow)
    logweights -= np.amax(logweights)
    weights = np.exp(logweights)
    
    # 2. Weighted Histogram
    # Note: Using fixed bins ensures all plots align on the x-axis
    hist, bin_edges = np.histogram(cmap, weights=weights, density=True, bins=30)
    
    # 3. Calculate Free Energy
    # Handle potential log(0) if sampling is poor in some bins
    with np.errstate(divide='ignore'):
        freeEnergy = -(1. / newBeta) * np.log(hist)
    
    # 4. Shift minimum to zero
    # We ignore 'inf' values when finding the minimum
    valid_indices = np.isfinite(freeEnergy)
    if np.any(valid_indices):
        freeEnergy -= np.min(freeEnergy[valid_indices])
    
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # 5. Plot
    # Get color corresponding to this temperature
    color = colormap(norm(newTemperature))
    
    # Only plot finite values to avoid artifacts
    ax.plot(bin_centers[valid_indices], freeEnergy[valid_indices], 
            linewidth=3, color=color)

# --- Aesthetics ---
ax.set_xlabel("Q", fontsize=30)
ax.set_ylabel("Free energy (kJ/mol)", fontsize=30)
ax.set_ylim(0, 41)
ax.set_yticks(np.arange(0, 42, 10))
ax.tick_params(axis='both', which='major', labelsize=25)
ax.set_xticks(np.arange(0, 1.01, 0.25))

# Add Colorbar to indicate Temperature
cbar = plt.colorbar(scalar_mappable, ax=ax)
cbar.set_label("Temperature (K)", fontsize=25)
cbar.ax.tick_params(labelsize=25)
cbar.set_ticks(np.arange(300, 490, 20)) # Optional: set specific ticks for cleanliness

plt.tight_layout()
plt.savefig("2B_linker_Reweighted_FreeEnergy_300-480K.png", dpi=400)
plt.show()

In [ ]:
# (Keeping your original loading logic intact)
ignore = 25000  # Ignore this many lines
# Initial load
COLVAR1 = np.genfromtxt("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/0/pasted_2000ns")[ignore:, :]
print(f"Replica 0 loaded: {len(COLVAR1)} frames")

for i in range(1, 16):  # Load data for replicas 1 to 15
    file_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/pasted_2000ns"
    # Concatenate subsequent files
    COLVAR1 = np.concatenate((COLVAR1, np.genfromtxt(file_path)[ignore:, :]))
    # print(f"Replica {i} loaded")

COLVAR = COLVAR1
time = np.copy(COLVAR[:, 0])
energy = np.copy(COLVAR[:, 1]) # Potential Energy
cmap = np.copy(COLVAR[:, 2])   # Collective Variable (Q)
bias = np.copy(COLVAR[:, 5])   # Bias Potential
rg = np.copy(COLVAR[:, 7]) 

# Constants
kb = 0.00831446  # kJ/mol/K
temperature = 390  # K
beta = 1. / (kb * temperature)
newTemperature = 300  # K
newBeta = 1. / (kb * newTemperature)

# Calculate weights
logweights = beta * bias + (beta - newBeta) * energy
logweights -= np.amax(logweights)  # Normalize to avoid large exponentials
weights = np.exp(logweights)

# Create a 2D histogram (free energy landscape)
H, xedges, yedges = np.histogram2d(cmap, rg, weights=weights, bins=(100, 100), range=[[0.01, 0.99], [0.01, 6]])
H = H.T  # Transpose for correct orientation
H = -(1 / newBeta) * np.log(gaussian_filter(H, sigma=3))  # Apply Gaussian filter and calculate free energy
H -= np.amin(H)  # Normalize the free energy

# Create meshgrid for plotting
X, Y = np.meshgrid(xedges, yedges)

# Plot the free energy landscape
plt.figure(figsize=(10, 8))
plt.pcolormesh(X, Y, H, vmin=0, vmax=40, cmap='viridis')  # Colormap with free energy
plt.colorbar(label="Free Energy (kJ/mol)")

# Contour overlay for free energy
x = (X[1:, 1:] + X[:-1, :-1]) / 2.
y = (Y[1:, 1:] + Y[:-1, :-1]) / 2.
plt.contour(x, y, H, np.arange(0, 40, 5), colors='black', linewidths=0.5)

# Plot limits and labels
plt.xlim([0.01, 0.99])
plt.ylim([0.7, 5.99])
plt.xticks(np.arange(0,1.01,0.25))
plt.yticks(np.arange(0.7,5.99, 1))
plt.xlabel("Q ")
plt.ylabel("Rg (nm)")
plt.tick_params(axis='x')
plt.tick_params(axis='y')
plt.tight_layout()
plt.savefig("2B_linker_FE_Q_Rg.png", dpi=400)
# Show the plot

plt.show()

In [ ]:
# (Keeping your original loading logic intact)
ignore = 25000  # Ignore this many lines
# Initial load
COLVAR1 = np.genfromtxt("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/0/pasted_2000ns")[ignore:, :]
print(f"Replica 0 loaded: {len(COLVAR1)} frames")

for i in range(1, 16):  # Load data for replicas 1 to 15
    file_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/pasted_2000ns"
    # Concatenate subsequent files
    COLVAR1 = np.concatenate((COLVAR1, np.genfromtxt(file_path)[ignore:, :]))
    # print(f"Replica {i} loaded")

COLVAR = COLVAR1
time = np.copy(COLVAR[:, 0])
energy = np.copy(COLVAR[:, 1]) # Potential Energy
cmap = np.copy(COLVAR[:, 2])   # Collective Variable (Q)
bias = np.copy(COLVAR[:, 5])   # Bias Potential
rg = np.copy(COLVAR[:, 7]) 

# Constants
kb = 0.00831446  # kJ/mol/K
temperature = 390  # K
beta = 1. / (kb * temperature)
newTemperature = 300  # K
newBeta = 1. / (kb * newTemperature)

# Calculate weights
logweights = beta * bias + (beta - newBeta) * energy
logweights -= np.amax(logweights)  # Normalize to avoid large exponentials
weights = np.exp(logweights)

# Create a 2D histogram (free energy landscape)
H, xedges, yedges = np.histogram2d(cmap, rg, weights=weights, bins=(100, 100), range=[[0.01, 0.99], [0.01, 6]])
H = H.T  # Transpose for correct orientation
H = -(1 / newBeta) * np.log(gaussian_filter(H, sigma=3))  # Apply Gaussian filter and calculate free energy
H -= np.amin(H)  # Normalize the free energy

# Create meshgrid for plotting
X, Y = np.meshgrid(xedges, yedges)

# Plot the free energy landscape
plt.figure(figsize=(10, 8))
plt.pcolormesh(X, Y, H, vmin=0, vmax=40, cmap='viridis')  # Colormap with free energy
plt.colorbar(label="Free Energy (kJ/mol)")

# Contour overlay for free energy
x = (X[1:, 1:] + X[:-1, :-1]) / 2.
y = (Y[1:, 1:] + Y[:-1, :-1]) / 2.
plt.contour(x, y, H, np.arange(0, 40, 5), colors='black', linewidths=0.5)

# Plot limits and labels
plt.xlim([0.01, 0.99])
plt.ylim([0.7, 5.0])

plt.xlabel("Q ")
plt.ylabel("Rg (nm)")
plt.tick_params(axis='x')
plt.tick_params(axis='y')

plt.xticks(np.arange(0,1.01,0.05), fontsize=10)
plt.yticks(np.arange(0.7,5.09, 0.1),fontsize=10)

y_values = [1.2,1.9]
for y in y_values:
    plt.axhline(y=y, color='red', linestyle='--', linewidth=3)

x_values = [0.55,0.73,0.77, 0.92]
for y in x_values:
    plt.axvline(x=y, color='red', linestyle='--', linewidth=3)
    
plt.tight_layout()
#plt.savefig("2B_linker_FE_Q_Rg.png", dpi=400)
# Show the plot

plt.show()


In [ ]:

with open("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/contact_map/2B_linker_contact_map", 'r') as f:
    data = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)
resid1=np.copy(data[:,0])
resid2=np.copy(data[:,1])
cm=np.copy(data[:,2])


# Define the grid based on the residue IDs
x = np.unique(resid1)
y = np.unique(resid2)

# Determine the number of residues
num_residues_x = len(x)
num_residues_y = len(y)
num_residues = max(num_residues_x, num_residues_y)
# Create an empty matrix to store the contact map
contact_map = np.zeros((num_residues_x, num_residues_y))

# Populate the contact map with the given data
for i in range(len(resid1)):
    row = np.where(x == resid1[i])[0][0]
    col = np.where(y == resid2[i])[0][0]
    contact_map[row, col] = cm[i]
    contact_map[col, row] = cm[i]  # Mirror image

        
# Reverse the gnuplot2 colormap
gnuplot2 = plt.cm.gnuplot2
gnuplot2_r = mcolors.ListedColormap(gnuplot2(np.linspace(1, 0, 256)))
# Plot
plt.figure(figsize=(12, 9))
plt.imshow(contact_map, extent=(x.min(), x.max(), y.max(), y.min()), cmap='gnuplot2_r')
plt.colorbar(label='Contact Map')
plt.xlabel('Residue ID 1', fontsize=30)
plt.ylabel('Residue ID 2', fontsize=30)
# plt.title('Contact Map')
# Reverse the y-axis
plt.ylim(y.max(), y.min())
y_values = [6, 55, 64, 113]
for y in y_values:
    plt.axhline(y=y, color='grey', linestyle='--', linewidth=1)
for y in y_values:
    plt.axvline(x=y, color='grey', linestyle='--', linewidth=1)
    
plt.xticks(np.arange(1, num_residues + 1, 10), fontsize=25)
plt.yticks(np.arange(1, num_residues + 1, 10), fontsize=25)
plt.xlim([1,num_residues])
plt.ylim([1,num_residues])
#plt.savefig("cmap_CHARMM_RRM_cutoff_6.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Load Data
file_path = "/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/contact_map/2B_linker_contact_map"

with open(file_path, 'r') as f:
    data = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)

resid1 = np.copy(data[:, 0])
resid2 = np.copy(data[:, 1])
cm = np.copy(data[:, 2])

# 2. Construct the Contact Map Matrix
x_coords = np.unique(resid1)
y_coords = np.unique(resid2)
num_residues_x = len(x_coords)
num_residues_y = len(y_coords)

contact_map = np.zeros((num_residues_x, num_residues_y))

for i in range(len(resid1)):
    row = np.where(x_coords == resid1[i])[0][0]
    col = np.where(y_coords == resid2[i])[0][0]
    contact_map[row, col] = cm[i]
    contact_map[col, row] = cm[i]

# 3. Define Regions and Calculate Averages in requested order
regions = {
    'Domain 1': (8, 55),
    'Linker': (56, 65),
    'Domain 2': (66, 111)
}

# Requested order: D1-Linker, D2-Linker, D1-D2
unique_pairs = [
    ('Domain 1', 'Linker'),
    ('Domain 2', 'Linker'),
    ('Domain 1', 'Domain 2')
]

pair_labels = [f"{p[0]}-{p[1]}" for p in unique_pairs]
averages = []

for reg1, reg2 in unique_pairs:
    start1, end1 = regions[reg1]
    start2, end2 = regions[reg2]
    
    indices1 = np.where((x_coords >= start1) & (x_coords <= end1))[0]
    indices2 = np.where((y_coords >= start2) & (y_coords <= end2))[0]
    
    if len(indices1) > 0 and len(indices2) > 0:
        sub_matrix = contact_map[np.ix_(indices1, indices2)]
        averages.append(np.mean(sub_matrix))
    else:
        averages.append(0.0)

# 4. Plotting
fig, ax = plt.subplots(figsize=(12, 7))

# Explicit color assignment: Black, Red, Green
bar_colors = ['black', 'red', 'green']
pair_labels = ['dmn1-linker', 'dmn2-linker', 'dmn1-dmn2']
bars = ax.bar(pair_labels, averages, color=bar_colors, alpha=0.9, width=0.6)

# Font Size and Label Settings
ax.set_ylabel('Average Contact Frequecny', fontsize=28)
ax.set_xlabel('Inter-Region Pair', fontsize=30)

# Requested fontsize 20 for xticks
ax.set_ylim(0, 0.065)
ax.set_yticks(np.arange(0, 0.08, 0.02)) # Stops at 0.06; use 0.08 to include 0.06 and headroom
ax.tick_params(axis='x', labelsize=30)
ax.tick_params(axis='y', labelsize=30)

plt.tight_layout()

# Save and Show
#plt.savefig("2BM_cltr1_dmn1_interdomain_contacts.png", dpi=400)
plt.show()

In [ ]:

with open("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/contact_map/cltr1_rc0.4_dmn1_Q_0.55-0.73_rg_1.2-1.9_contact_map", 'r') as f:
    data = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)
resid1=np.copy(data[:,0])
resid2=np.copy(data[:,1])
cm=np.copy(data[:,2])


# Define the grid based on the residue IDs
x = np.unique(resid1)
y = np.unique(resid2)

# Determine the number of residues
num_residues_x = len(x)
num_residues_y = len(y)
num_residues = max(num_residues_x, num_residues_y)
# Create an empty matrix to store the contact map
contact_map = np.zeros((num_residues_x, num_residues_y))

# Populate the contact map with the given data
for i in range(len(resid1)):
    row = np.where(x == resid1[i])[0][0]
    col = np.where(y == resid2[i])[0][0]
    contact_map[row, col] = cm[i]
    contact_map[col, row] = cm[i]  # Mirror image

        
# Reverse the gnuplot2 colormap
gnuplot2 = plt.cm.gnuplot2
gnuplot2_r = mcolors.ListedColormap(gnuplot2(np.linspace(1, 0, 256)))
# Plot
plt.figure(figsize=(12, 9))
plt.imshow(contact_map, extent=(x.min(), x.max(), y.max(), y.min()), cmap='gnuplot2_r')
plt.colorbar(label='Contact Map')
plt.xlabel('Residue ID 1', fontsize=30)
plt.ylabel('Residue ID 2', fontsize=30)
# plt.title('Contact Map')
# Reverse the y-axis
plt.ylim(y.max(), y.min())
y_values = [6, 55, 64, 113]
for y in y_values:
    plt.axhline(y=y, color='grey', linestyle='--', linewidth=1)
for y in y_values:
    plt.axvline(x=y, color='grey', linestyle='--', linewidth=1)
    
plt.xticks(np.arange(1, num_residues + 1, 10), fontsize=25)
plt.yticks(np.arange(1, num_residues + 1, 10), fontsize=25)
plt.xlim([1,num_residues])
plt.ylim([1,num_residues])
#plt.savefig("cmap_CHARMM_RRM_cutoff_6.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the regions based on your residue ranges
regions = {
    'Domain 1': (8, 55),
    'Linker': (56, 65),
    'Domain 2': (66, 111)
}
labels = list(regions.keys())
n_regions = len(labels)

# Calculate the average contact between regions
avg_matrix = np.zeros((n_regions, n_regions))

for i, (name_i, (start_i, end_i)) in enumerate(regions.items()):
    for j, (name_j, (start_j, end_j)) in enumerate(regions.items()):
        # Slicing the contact_map (assuming contact_map variable exists from your previous code)
        # We use start-1 to convert 1-based residue IDs to 0-based indices
        sub_matrix = contact_map[start_i-1:end_i, start_j-1:end_j]
        avg_matrix[i, j] = np.mean(sub_matrix)

# Set up the grouped bar chart
x_indices = np.arange(len(labels))
width = 0.25  # Width of each bar

fig, ax = plt.subplots(figsize=(10, 6))

# Define a color palette
colors = ['#440154', '#21918c', '#fde725'] # Viridis-like palette

# Plot bars for each interaction pair
for i in range(n_regions):
    # offset groups the bars around the x-tick
    offset = (i - 1) * width
    ax.bar(x_indices + offset, avg_matrix[:, i], width, label=f'with {labels[i]}', color=colors[i], edgecolor='black')

# Formatting the plot
ax.set_ylabel('Average Contact Frequency', fontsize=25)
ax.set_xlabel('Reference Region', fontsize=25)
ax.set_xticks(x_indices)
ax.set_xticklabels(labels, fontsize=25)
ax.legend(title="Contacting Region", fontsize=15, title_fontsize=15)

# Add value labels on top of each bar for clarity
for i in range(n_regions): # Target
    for j in range(n_regions): # Source
        val = avg_matrix[j, i]
        ax.text(j + (i-1)*width, val + 0.005, f'{val:.3f}', 
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.ylim(0, np.max(avg_matrix) * 1.2) # Add some head room for labels
plt.tight_layout()
#plt.savefig("domain_interaction_bars.png", dpi=400)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Define the regions and color mapping
regions = {
    'Domain 1': (8, 55),
    'Linker': (56, 65),
    'Domain 2': (66, 111)
}
region_names = list(regions.keys())
color_map = {
    'Domain 1': 'blue',
    'Linker': 'green',
    'Domain 2': 'purple'
}

# 1. Calculate Average Contact Matrix
# We ensure indices are correctly mapped to the unique residue IDs (x) from your code
avg_matrix = np.zeros((3, 3))
for i, name_i in enumerate(region_names):
    start_i, end_i = regions[name_i]
    indices_i = np.where((x >= start_i) & (x <= end_i))[0]
    for j, name_j in enumerate(region_names):
        start_j, end_j = regions[name_j]
        indices_j = np.where((x >= start_j) & (x <= end_j))[0]
        
        if len(indices_i) > 0 and len(indices_j) > 0:
            # np.ix_ allows for rectangular slicing of the matrix
            sub_matrix = contact_map[np.ix_(indices_i, indices_j)]
            avg_matrix[i, j] = np.mean(sub_matrix)

# 2. Plotting the Bar Chart excluding self-contacts (diagonal of the matrix)
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(len(region_names))
width = 0.3

for i, source in enumerate(region_names):
    # Filter out self-interaction (diagonal)
    targets = [t for t in region_names if t != source]
    # Two bars per group: positions shifted from the center tick
    offsets = [-width/2, width/2]
    
    for k, target in enumerate(targets):
        j = region_names.index(target)
        val = avg_matrix[i, j]
        
        # Color corresponds to the 'target' region
        ax.bar(x_pos[i] + offsets[k], val, width, 
               color=color_map[target], edgecolor='black', alpha=0.85)
        
        # Add value annotation above bars
        ax.text(x_pos[i] + offsets[k], val + 0.002, f'{val:.3f}', 
                ha='center', va='bottom', fontsize=10, fontweight='bold')

# 3. Aesthetics and Formatting
ax.set_xticks(x_pos)
ax.set_xticklabels(region_names, fontsize=20)
ax.set_ylabel('Average Contact Frequency', fontsize=20)
ax.set_xlabel('Source Region', fontsize=20)
#ax.set_title('Inter-Domain Contact Frequencies', fontsize=16)

# Custom Legend to clarify color coding
legend_elements = [Line2D([0], [0], color=color_map[name], lw=4, label=name) for name in region_names]
ax.legend(handles=legend_elements, title="Interacting with", fontsize=15, loc='upper right')

plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
#plt.savefig("interdomain_contacts_bar.png", dpi=400)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Load Data
file_path = "/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/contact_map/cltr1_rc0.4_dmn1_Q_0.55-0.73_rg_1.2-1.9_contact_map"

with open(file_path, 'r') as f:
    data = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)

resid1 = np.copy(data[:, 0])
resid2 = np.copy(data[:, 1])
cm = np.copy(data[:, 2])

# 2. Construct the Contact Map Matrix
x_coords = np.unique(resid1)
y_coords = np.unique(resid2)
num_residues_x = len(x_coords)
num_residues_y = len(y_coords)

contact_map = np.zeros((num_residues_x, num_residues_y))

for i in range(len(resid1)):
    row = np.where(x_coords == resid1[i])[0][0]
    col = np.where(y_coords == resid2[i])[0][0]
    contact_map[row, col] = cm[i]
    contact_map[col, row] = cm[i]

# 3. Define Regions and Calculate Averages in requested order
regions = {
    'Domain 1': (8, 55),
    'Linker': (56, 65),
    'Domain 2': (66, 111)
}

# Requested order: D1-Linker, D2-Linker, D1-D2
unique_pairs = [
    ('Domain 1', 'Linker'),
    ('Domain 2', 'Linker'),
    ('Domain 1', 'Domain 2')
]

pair_labels = [f"{p[0]}-{p[1]}" for p in unique_pairs]
averages = []

for reg1, reg2 in unique_pairs:
    start1, end1 = regions[reg1]
    start2, end2 = regions[reg2]
    
    indices1 = np.where((x_coords >= start1) & (x_coords <= end1))[0]
    indices2 = np.where((y_coords >= start2) & (y_coords <= end2))[0]
    
    if len(indices1) > 0 and len(indices2) > 0:
        sub_matrix = contact_map[np.ix_(indices1, indices2)]
        averages.append(np.mean(sub_matrix))
    else:
        averages.append(0.0)

# 4. Plotting
fig, ax = plt.subplots(figsize=(12, 7))

# Explicit color assignment: Black, Red, Green
bar_colors = ['black', 'red', 'green']
pair_labels = ['dmn1-linker', 'dmn2-linker', 'dmn1-dmn2']
bars = ax.bar(pair_labels, averages, color=bar_colors, alpha=0.9, width=0.6)

# Font Size and Label Settings
ax.set_ylabel('Average Contact Frequecny', fontsize=28)
ax.set_xlabel('Inter-Region Pair', fontsize=30)

# Requested fontsize 20 for xticks
ax.set_ylim(0, 0.065)
ax.set_yticks(np.arange(0, 0.08, 0.02)) # Stops at 0.06; use 0.08 to include 0.06 and headroom
ax.tick_params(axis='x', labelsize=30)
ax.tick_params(axis='y', labelsize=30)

plt.tight_layout()

# Save and Show
plt.savefig("2BM_cltr1_dmn1_interdomain_contacts.png", dpi=400)
plt.show()

In [ ]:

with open("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/contact_map/cltr1_rc0.4_Q_0.77-0.92_rg_1.2-1.9_contact_map", 'r') as f:
    data = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)
resid1=np.copy(data[:,0])
resid2=np.copy(data[:,1])
cm=np.copy(data[:,2])


# Define the grid based on the residue IDs
x = np.unique(resid1)
y = np.unique(resid2)

# Determine the number of residues
num_residues_x = len(x)
num_residues_y = len(y)
num_residues = max(num_residues_x, num_residues_y)
# Create an empty matrix to store the contact map
contact_map = np.zeros((num_residues_x, num_residues_y))

# Populate the contact map with the given data
for i in range(len(resid1)):
    row = np.where(x == resid1[i])[0][0]
    col = np.where(y == resid2[i])[0][0]
    contact_map[row, col] = cm[i]
    contact_map[col, row] = cm[i]  # Mirror image

        
# Reverse the gnuplot2 colormap
gnuplot2 = plt.cm.gnuplot2
gnuplot2_r = mcolors.ListedColormap(gnuplot2(np.linspace(1, 0, 256)))
# Plot
plt.figure(figsize=(12, 9))
plt.imshow(contact_map, extent=(x.min(), x.max(), y.max(), y.min()), cmap='gnuplot2_r')
plt.colorbar(label='Contact Map')
plt.xlabel('Residue ID 1', fontsize=30)
plt.ylabel('Residue ID 2', fontsize=30)
# plt.title('Contact Map')
# Reverse the y-axis
plt.ylim(y.max(), y.min())
y_values = [6, 55, 64, 113]
for y in y_values:
    plt.axhline(y=y, color='grey', linestyle='--', linewidth=1)
for y in y_values:
    plt.axvline(x=y, color='grey', linestyle='--', linewidth=1)
    
plt.xticks(np.arange(1, num_residues + 1, 10), fontsize=25)
plt.yticks(np.arange(1, num_residues + 1, 10), fontsize=25)
plt.xlim([1,num_residues])
plt.ylim([1,num_residues])
#plt.savefig("cmap_CHARMM_RRM_cutoff_6.png", dpi=300)
plt.show()

In [ ]:

with open("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/contact_map/cltr2_rc0.4_Q_0.77-0.92_rg_1.2-1.9_contact_map", 'r') as f:
    data = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)
resid1=np.copy(data[:,0])
resid2=np.copy(data[:,1])
cm=np.copy(data[:,2])


# Define the grid based on the residue IDs
x = np.unique(resid1)
y = np.unique(resid2)

# Determine the number of residues
num_residues_x = len(x)
num_residues_y = len(y)
num_residues = max(num_residues_x, num_residues_y)
# Create an empty matrix to store the contact map
contact_map = np.zeros((num_residues_x, num_residues_y))

# Populate the contact map with the given data
for i in range(len(resid1)):
    row = np.where(x == resid1[i])[0][0]
    col = np.where(y == resid2[i])[0][0]
    contact_map[row, col] = cm[i]
    contact_map[col, row] = cm[i]  # Mirror image

        
# Reverse the gnuplot2 colormap
gnuplot2 = plt.cm.gnuplot2
gnuplot2_r = mcolors.ListedColormap(gnuplot2(np.linspace(1, 0, 256)))
# Plot
plt.figure(figsize=(12, 9))
plt.imshow(contact_map, extent=(x.min(), x.max(), y.max(), y.min()), cmap='gnuplot2_r')
plt.colorbar(label='Contact Map')
plt.xlabel('Residue ID 1', fontsize=30)
plt.ylabel('Residue ID 2', fontsize=30)
# plt.title('Contact Map')
# Reverse the y-axis
plt.ylim(y.max(), y.min())
y_values = [6, 55, 64, 113]
for y in y_values:
    plt.axhline(y=y, color='grey', linestyle='--', linewidth=1)
for y in y_values:
    plt.axvline(x=y, color='grey', linestyle='--', linewidth=1)
    
plt.xticks(np.arange(1, num_residues + 1, 10), fontsize=25)
plt.yticks(np.arange(1, num_residues + 1, 10), fontsize=25)
plt.xlim([1,num_residues])
plt.ylim([1,num_residues])
#plt.savefig("cmap_CHARMM_RRM_cutoff_6.png", dpi=300)
plt.show()

In [ ]:

with open("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/contact_map/2B_linker_contact_map_all_nds_prot_Q_0.55-0.73_rg_1.2-1.9", 'r') as f:
    data = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)
resid1=np.copy(data[:,0])
resid2=np.copy(data[:,1])
cm=np.copy(data[:,2])


# Define the grid based on the residue IDs
x = np.unique(resid1)
y = np.unique(resid2)

# Determine the number of residues
num_residues_x = len(x)
num_residues_y = len(y)
num_residues = max(num_residues_x, num_residues_y)
# Create an empty matrix to store the contact map
contact_map = np.zeros((num_residues_x, num_residues_y))

# Populate the contact map with the given data
for i in range(len(resid1)):
    row = np.where(x == resid1[i])[0][0]
    col = np.where(y == resid2[i])[0][0]
    contact_map[row, col] = cm[i]
    contact_map[col, row] = cm[i]  # Mirror image

        
# Reverse the gnuplot2 colormap
gnuplot2 = plt.cm.gnuplot2
gnuplot2_r = mcolors.ListedColormap(gnuplot2(np.linspace(1, 0, 256)))
# Plot
plt.figure(figsize=(12, 9))
plt.imshow(contact_map, extent=(x.min(), x.max(), y.max(), y.min()), cmap='gnuplot2_r')
plt.colorbar(label='Contact Map')
plt.xlabel('Residue ID 1', fontsize=30)
plt.ylabel('Residue ID 2', fontsize=30)
# plt.title('Contact Map')
# Reverse the y-axis
plt.ylim(y.max(), y.min())
y_values = [6, 55, 64, 113]
for y in y_values:
    plt.axhline(y=y, color='grey', linestyle='--', linewidth=1)
for y in y_values:
    plt.axvline(x=y, color='grey', linestyle='--', linewidth=1)
    
plt.xticks(np.arange(1, num_residues + 1, 10), fontsize=25)
plt.yticks(np.arange(1, num_residues + 1, 10), fontsize=25)
plt.xlim([1,num_residues])
plt.ylim([1,num_residues])
#plt.savefig("cmap_CHARMM_RRM_cutoff_6.png", dpi=300)
plt.show()

# nd 5 subtrajectoris- Q~0.8

In [ ]:

with open("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/contact_map/2B_linker_contact_map_nd5", 'r') as f:
    data = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)
resid1=np.copy(data[:,0])
resid2=np.copy(data[:,1])
cm=np.copy(data[:,2])


# Define the grid based on the residue IDs
x = np.unique(resid1)
y = np.unique(resid2)

# Determine the number of residues
num_residues_x = len(x)
num_residues_y = len(y)
num_residues = max(num_residues_x, num_residues_y)
# Create an empty matrix to store the contact map
contact_map = np.zeros((num_residues_x, num_residues_y))

# Populate the contact map with the given data
for i in range(len(resid1)):
    row = np.where(x == resid1[i])[0][0]
    col = np.where(y == resid2[i])[0][0]
    contact_map[row, col] = cm[i]
    contact_map[col, row] = cm[i]  # Mirror image

        
# Reverse the gnuplot2 colormap
gnuplot2 = plt.cm.gnuplot2
gnuplot2_r = mcolors.ListedColormap(gnuplot2(np.linspace(1, 0, 256)))
# Plot
plt.figure(figsize=(12, 9))
plt.imshow(contact_map, extent=(x.min(), x.max(), y.max(), y.min()), cmap='gnuplot2_r')
plt.colorbar(label='Contact Map')
plt.xlabel('Residue ID 1', fontsize=30)
plt.ylabel('Residue ID 2', fontsize=30)
# plt.title('Contact Map')
# Reverse the y-axis
plt.ylim(y.max(), y.min())
y_values = [6, 55, 64, 113]
for y in y_values:
    plt.axhline(y=y, color='grey', linestyle='--', linewidth=1)
for y in y_values:
    plt.axvline(x=y, color='grey', linestyle='--', linewidth=1)
    
plt.xticks(np.arange(1, num_residues + 1, 10), fontsize=25)
plt.yticks(np.arange(1, num_residues + 1, 10), fontsize=25)
plt.xlim([1,num_residues])
plt.ylim([1,num_residues])
#plt.savefig("cmap_CHARMM_RRM_cutoff_6.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Configuration
base_path = "/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/contact_map/"
stride = 100000
total_segments = 20
rows, cols = 4, 5
y_markers = [6, 55, 64, 113]

# Setup Colormap
gnuplot2 = plt.cm.gnuplot2
gnuplot2_r = mcolors.ListedColormap(gnuplot2(np.linspace(1, 0, 256)))

# Initialize the figure
fig, axs = plt.subplots(rows, cols, figsize=(22, 16), sharex=True, sharey=True)
axs = axs.flatten()

# DYNAMIC SIZE: Let's find the max residue ID across all files first 
# or set it slightly higher than your known last residue.
max_res_found = 0

for i in range(total_segments):
    start = i * stride
    end = (i + 1) * stride
    file_name = f"subtrj_nd5_{start}_{end}_contact_map"
    full_path = f"{base_path}{file_name}"
    
    try:
        with open(full_path, 'r') as f:
            data = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)
        
        resid1 = data[:, 0].astype(int)
        resid2 = data[:, 1].astype(int)
        cm = data[:, 2]

        # Safety: Update max_res if we find a larger ID
        current_max = max(resid1.max(), resid2.max())
        if current_max > max_res_found:
            max_res_found = current_max

        # We initialize a large enough matrix (adding 1 for safety)
        # Using a fixed size for all plots is better for visual comparison
        # Let's use 120 or the max found
        matrix_size = max(116, int(max_res_found)) 
        contact_matrix = np.zeros((matrix_size, matrix_size))

        # MIRRORING with index safety
        idx1 = resid1 - 1
        idx2 = resid2 - 1
        
        contact_matrix[idx1, idx2] = cm
        contact_matrix[idx2, idx1] = cm

        # Plot
        im = axs[i].imshow(contact_matrix, origin='lower', cmap='gnuplot2_r', 
                           extent=(1, matrix_size, 1, matrix_size), vmin=0, vmax=1)
        
        axs[i].set_title(f"{start/1000:.0f}-{end/1000:.0f} ns", fontsize=30)
        
        for val in y_markers:
            axs[i].axhline(y=val, color='grey', linestyle='--', linewidth=0.6, alpha=0.5)
            axs[i].axvline(x=val, color='grey', linestyle='--', linewidth=0.6, alpha=0.5)

    except FileNotFoundError:
        axs[i].text(0.5, 0.5, "File Not Found", ha='center')

# Final formatting
for ax in axs:
    ax.set_xlim(1, 116) # Adjust these to zoom in on your protein specifically
    ax.set_ylim(1, 116)

fig.supxlabel('Residue ID 1', fontsize=35)
fig.supylabel('Residue ID 2', fontsize=35)

# Colorbar fix (cax instead of caxes)
cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7]) 
fig.colorbar(im, cax=cbar_ax, label='Contact Probability')

plt.subplots_adjust(right=0.91, hspace=0.3, wspace=0.1)
plt.savefig("2B_linker_nd5_split_cmap.png", dpi=400)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Configuration
base_path = "/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/contact_map/"
stride = 100000
total_segments = 10
linker_start, linker_end = 55, 64
max_res = 116 # Ensure this covers your total residue count

# Setup Colormap
gnuplot2 = plt.cm.gnuplot2
gnuplot2_r = mcolors.ListedColormap(gnuplot2(np.linspace(1, 0, 256)))

# 20 rows, 1 col.
fig, axs = plt.subplots(10, 1, figsize=(10, 15), sharex=True)
axs = axs.flatten()

# Track if we actually successfully loaded anything
plot_triggered = False

for i in range(total_segments):
    start = i * stride
    end = (i + 1) * stride
    file_name = f"subtrj_nd5_{start}_{end}_contact_map"
    full_path = f"{base_path}{file_name}"
    
    try:
        # Fixed Loader: Filtering lines manually to avoid the tuple error
        with open(full_path, 'r') as f:
            data = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)
        
        if data.size == 0 or len(data.shape) < 2:
            print(f"Warning: Segment {i} file is empty or malformed.")
            continue

        resid1 = data[:, 0].astype(int)
        resid2 = data[:, 1].astype(int)
        cm = data[:, 2]

        # Use a consistent matrix size for all subplots
        # N should be at least max_res or the highest ID in the data
        N = max(max_res, int(resid1.max()), int(resid2.max()))
        full_matrix = np.zeros((N, N))

        # Mirroring
        full_matrix[resid1-1, resid2-1] = cm
        full_matrix[resid2-1, resid1-1] = cm

        # SLICE: Rows 55-64 (indices 54 to 64)
        # SLICE rows AND mirrored columns
        row_slice = full_matrix[linker_start-1:linker_end, :]
        col_slice = full_matrix[:, linker_start-1:linker_end].T

        # Combine to preserve mirror symmetry
        linker_slice = np.maximum(row_slice, col_slice)
        
        # Plot
        im = axs[i].imshow(linker_slice, origin='lower', cmap='gnuplot2_r', 
                           extent=(1, N, linker_start, linker_end), 
                           vmin=0, vmax=1, aspect='auto')
        
        axs[i].set_title(f"{start/1000:.0f}-{end/1000:.0f} ns",fontsize=15)
        axs[i].set_yticks([linker_start, linker_end])
        axs[i].tick_params(axis='y', labelsize=20)
        
        # Vertical markers for domain boundaries
        for val in [6, 18, 24, 37, 42, 55, 64, 76, 82, 95, 100, 113]:
            axs[i].axvline(x=val, color='black', linestyle='--', linewidth=2, alpha=0.3)
        
        plot_triggered = True

    except Exception as e:
        print(f"Error at segment {i}: {e}")
        axs[i].text(0.5, 0.5, "Data Error", ha='center')

# Final Labels
axs[-1].set_xlabel('Residue ID', fontsize=25)
axs[-1].tick_params(axis='x', labelsize=20)
# Colorbar - Using a safer approach
if plot_triggered:
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    # Use the figure directly to add the colorbar axis
    cbar_ax = fig.add_axes([0.25, 0.03, 0.5, 0.01]) 
    fig.colorbar(im, cax=cbar_ax, orientation='horizontal', label='Contact Probability')
else:
    print("No data was successfully plotted.")

plt.savefig("2B_cmap_nd15_subtrj1", dpi=400)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Configuration
base_path = "/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/contact_map/"
stride = 100000
offset = 10              # <-- START FROM SEGMENT 10
n_segments = 10          # <-- PLOT 10 SEGMENTS
linker_start, linker_end = 55, 64
max_res = 116

# Colormap
gnuplot2 = plt.cm.gnuplot2
gnuplot2_r = mcolors.ListedColormap(gnuplot2(np.linspace(1, 0, 256)))

# Figure
fig, axs = plt.subplots(10, 1, figsize=(10, 15), sharex=True)
axs = axs.flatten()

plot_triggered = False

for i in range(n_segments):
    i_global = i + offset        # <-- GLOBAL SEGMENT INDEX

    start = i_global * stride
    end = (i_global + 1) * stride

    file_name = f"subtrj_nd5_{start}_{end}_contact_map"
    full_path = f"{base_path}{file_name}"

    try:
        with open(full_path, 'r') as f:
            data = np.genfromtxt(
                (line for line in f if not line.startswith(('#', '@'))),
                dtype=float
            )

        if data.size == 0 or len(data.shape) < 2:
            print(f"Warning: Segment {i_global} file is empty or malformed.")
            continue

        resid1 = data[:, 0].astype(int)
        resid2 = data[:, 1].astype(int)
        cm = data[:, 2]

        N = max(max_res, resid1.max(), resid2.max())
        full_matrix = np.zeros((N, N))

        # Mirror
        full_matrix[resid1-1, resid2-1] = cm
        full_matrix[resid2-1, resid1-1] = cm

        # Linker slice + mirror
        row_slice = full_matrix[linker_start-1:linker_end, :]
        col_slice = full_matrix[:, linker_start-1:linker_end].T
        linker_slice = np.maximum(row_slice, col_slice)

        im = axs[i].imshow(
            linker_slice,
            origin='lower',
            cmap='gnuplot2_r',
            extent=(1, N, linker_start, linker_end),
            vmin=0,
            vmax=1,
            aspect='auto'
        )

        axs[i].set_title(f"{start/1000:.0f}-{end/1000:.0f} ns", fontsize=15)
        axs[i].set_yticks([linker_start, linker_end])
        axs[i].tick_params(axis='y', labelsize=20)

        for val in [6, 18, 24, 37, 42, 55, 64, 76, 82, 95, 100, 113]:
            axs[i].axvline(val, color='black', linestyle='--',
                           linewidth=2, alpha=0.3)

        plot_triggered = True

    except Exception as e:
        print(f"Error at segment {i_global}: {e}")
        axs[i].text(0.5, 0.5, "Data Error", ha='center')

# Labels
axs[-1].set_xlabel('Residue ID', fontsize=25)
axs[-1].tick_params(axis='x', labelsize=20)

# Colorbar
if plot_triggered:
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    cbar_ax = fig.add_axes([0.25, 0.03, 0.5, 0.01])
    fig.colorbar(im, cax=cbar_ax, orientation='horizontal',
                 label='Contact Probability')

plt.savefig("2B_cmap_nd15_subtrj2", dpi=400)
plt.show()


# nd14- Q~0.8

In [ ]:

with open("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/contact_map/2B_linker_contact_map_nd14", 'r') as f:
    data = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)
resid1=np.copy(data[:,0])
resid2=np.copy(data[:,1])
cm=np.copy(data[:,2])


# Define the grid based on the residue IDs
x = np.unique(resid1)
y = np.unique(resid2)

# Determine the number of residues
num_residues_x = len(x)
num_residues_y = len(y)
num_residues = max(num_residues_x, num_residues_y)
# Create an empty matrix to store the contact map
contact_map = np.zeros((num_residues_x, num_residues_y))

# Populate the contact map with the given data
for i in range(len(resid1)):
    row = np.where(x == resid1[i])[0][0]
    col = np.where(y == resid2[i])[0][0]
    contact_map[row, col] = cm[i]
    contact_map[col, row] = cm[i]  # Mirror image

        
# Reverse the gnuplot2 colormap
gnuplot2 = plt.cm.gnuplot2
gnuplot2_r = mcolors.ListedColormap(gnuplot2(np.linspace(1, 0, 256)))
# Plot
plt.figure(figsize=(12, 9))
plt.imshow(contact_map, extent=(x.min(), x.max(), y.max(), y.min()), cmap='gnuplot2_r')
plt.colorbar(label='Contact Map')
plt.xlabel('Residue ID 1', fontsize=30)
plt.ylabel('Residue ID 2', fontsize=30)
# plt.title('Contact Map')
# Reverse the y-axis
plt.ylim(y.max(), y.min())
y_values = [6, 55, 64, 113]
for y in y_values:
    plt.axhline(y=y, color='grey', linestyle='--', linewidth=1)
for y in y_values:
    plt.axvline(x=y, color='grey', linestyle='--', linewidth=1)
    
plt.xticks(np.arange(1, num_residues + 1, 10), fontsize=25)
plt.yticks(np.arange(1, num_residues + 1, 10), fontsize=25)
plt.xlim([1,num_residues])
plt.ylim([1,num_residues])
#plt.savefig("cmap_CHARMM_RRM_cutoff_6.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Configuration
base_path = "/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/contact_map/"
stride = 100000
total_segments = 20
rows, cols = 4, 5
y_markers = [6, 55, 64, 113]

# Setup Colormap
gnuplot2 = plt.cm.gnuplot2
gnuplot2_r = mcolors.ListedColormap(gnuplot2(np.linspace(1, 0, 256)))

# Initialize the figure
fig, axs = plt.subplots(rows, cols, figsize=(22, 16), sharex=True, sharey=True)
axs = axs.flatten()

# DYNAMIC SIZE: Let's find the max residue ID across all files first 
# or set it slightly higher than your known last residue.
max_res_found = 0

for i in range(total_segments):
    start = i * stride
    end = (i + 1) * stride
    file_name = f"subtrj_nd14_{start}_{end}_contact_map"
    full_path = f"{base_path}{file_name}"
    
    try:
        with open(full_path, 'r') as f:
            data = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)
        
        resid1 = data[:, 0].astype(int)
        resid2 = data[:, 1].astype(int)
        cm = data[:, 2]

        # Safety: Update max_res if we find a larger ID
        current_max = max(resid1.max(), resid2.max())
        if current_max > max_res_found:
            max_res_found = current_max

        # We initialize a large enough matrix (adding 1 for safety)
        # Using a fixed size for all plots is better for visual comparison
        # Let's use 120 or the max found
        matrix_size = max(116, int(max_res_found)) 
        contact_matrix = np.zeros((matrix_size, matrix_size))

        # MIRRORING with index safety
        idx1 = resid1 - 1
        idx2 = resid2 - 1
        
        contact_matrix[idx1, idx2] = cm
        contact_matrix[idx2, idx1] = cm

        # Plot
        im = axs[i].imshow(contact_matrix, origin='lower', cmap='gnuplot2_r', 
                           extent=(1, matrix_size, 1, matrix_size), vmin=0, vmax=1)
        
        axs[i].set_title(f"{start/1000:.0f}-{end/1000:.0f} ns", fontsize=30)
        
        for val in y_markers:
            axs[i].axhline(y=val, color='grey', linestyle='--', linewidth=0.6, alpha=0.5)
            axs[i].axvline(x=val, color='grey', linestyle='--', linewidth=0.6, alpha=0.5)

    except FileNotFoundError:
        axs[i].text(0.5, 0.5, "File Not Found", ha='center')

# Final formatting
for ax in axs:
    ax.set_xlim(1, 116) # Adjust these to zoom in on your protein specifically
    ax.set_ylim(1, 116)

fig.supxlabel('Residue ID 1', fontsize=35)
fig.supylabel('Residue ID 2', fontsize=35)

# Colorbar fix (cax instead of caxes)
cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7]) 
fig.colorbar(im, cax=cbar_ax, label='Contact Probability')

plt.subplots_adjust(right=0.91, hspace=0.3, wspace=0.1)
plt.savefig("2B_linker_nd14_split_cmap.png", dpi=400)
plt.show()

In [ ]:
ignore = 25000
num_replicas = 16

# --- 1. Load and Process RAMA Data ---
# Load the full concatenated file
rama_file = '/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/rama/rama_turn1_PRO-20.xvg'
with open(rama_file, 'r') as 
    raw_rama = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)

# Calculate frames per replica to verify consistency
total_rama_frames = raw_rama.shape[0]
frames_per_replica = total_rama_frames // num_replicas

if total_rama_frames % num_replicas != 0:
    print(f"Warning: Total rama frames ({total_rama_frames}) is not divisible by {num_replicas}!")

# Reshape into (16, frames_per_replica, columns)
rama_reshaped = raw_rama.reshape((num_replicas, frames_per_replica, raw_rama.shape[1]))

# Slice off the 'ignore' frames from the second dimension (axis 1)
rama_sliced = rama_reshaped[:, ignore:, :]

# Flatten back to 2D array: (Total_Effective_Frames, columns)
rama_final = rama_sliced.reshape((-1, raw_rama.shape[1]))

# Extract Phi and Psi
phi = rama_final[:, 0]
psi = rama_final[:, 1]

COLVAR1 = np.genfromtxt("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/0/pasted_2000ns")[ignore:, :]
print(f"Replica 0 loaded: {len(COLVAR1)} frames")

for i in range(1, 16):  # Load data for replicas 1 to 15
    file_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/pasted_2000ns"
    # Concatenate subsequent files
    COLVAR1 = np.concatenate((COLVAR1, np.genfromtxt(file_path)[ignore:, :]))
    # print(f"Replica {i} loaded")

COLVAR = COLVAR1
time = np.copy(COLVAR[:, 0])
energy = np.copy(COLVAR[:, 1]) # Potential Energy
#cmap = np.copy(COLVAR[:, 2])   # Collective Variable (Q)
bias = np.copy(COLVAR[:, 5])   # Bias Potential
#rg = np.copy(COLVAR[:, 7]) 

phi_AF= -80.2232
psi_AF= -19.5668
print (len(phi), len (time), time [-1], time[0], time[1]) 
kb=0.00831446 # kJ/mol/K
temperature=390 # K
beta=1./(kb*temperature)
newTemperature=300 # K
newBeta=1./(kb*newTemperature)
logweights=beta*bias+(beta-newBeta)*energy
logweights -= np.amax(logweights)
weights=np.exp(logweights)


H1, xedges1, yedges1 = np.histogram2d(phi, psi, weights=weights,bins=(100,100),range=[[-200,200],[-200,200]])
H1 = H1.T
H1=-(1/newBeta)*np.log(gaussian_filter(H1, sigma=3))
H1 -= np.amin(H1) 
X1, Y1 = np.meshgrid(xedges1, yedges1)
cmap = plt.cm.hot
plt.pcolormesh(X1, Y1, H1, vmin=0,vmax=15, cmap=cmap)
plt.colorbar()
x = (X1[1:,1:]+X1[:-1,:-1])/2.
y = (Y1[1:,1:]+Y1[:-1,:-1])/2.
plt.contour(x,y, H1 ,np.arange(0,25,5),colors='black',linewidths=0.5)

plt.xlabel("Phi (ϕ) Angle (degrees)", fontsize=20)
plt.ylabel("Psi (ψ) Angle (degrees)",fontsize=20)
plt.tick_params(axis='x', labelsize=20)
plt.tick_params(axis='y', labelsize=20)
plt.scatter(phi_AF, psi_AF, color='blue')

plt.show()

In [ ]:
ignore = 25000
num_replicas = 16

# --- 1. Load and Process RAMA Data ---
# Load the full concatenated file
rama_file = '/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/rama/rama_turn1-ASN-21.xvg'
with open(rama_file, 'r') as f:
    raw_rama = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)

# Calculate frames per replica to verify consistency
total_rama_frames = raw_rama.shape[0]
frames_per_replica = total_rama_frames // num_replicas

if total_rama_frames % num_replicas != 0:
    print(f"Warning: Total rama frames ({total_rama_frames}) is not divisible by {num_replicas}!")

# Reshape into (16, frames_per_replica, columns)
rama_reshaped = raw_rama.reshape((num_replicas, frames_per_replica, raw_rama.shape[1]))

# Slice off the 'ignore' frames from the second dimension (axis 1)
rama_sliced = rama_reshaped[:, ignore:, :]

# Flatten back to 2D array: (Total_Effective_Frames, columns)
rama_final = rama_sliced.reshape((-1, raw_rama.shape[1]))

# Extract Phi and Psi
phi = rama_final[:, 0]
psi = rama_final[:, 1]

COLVAR1 = np.genfromtxt("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/0/pasted_2000ns")[ignore:, :]
print(f"Replica 0 loaded: {len(COLVAR1)} frames")

for i in range(1, 16):  # Load data for replicas 1 to 15
    file_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/pasted_2000ns"
    # Concatenate subsequent files
    COLVAR1 = np.concatenate((COLVAR1, np.genfromtxt(file_path)[ignore:, :]))
    # print(f"Replica {i} loaded")

COLVAR = COLVAR1
time = np.copy(COLVAR[:, 0])
energy = np.copy(COLVAR[:, 1]) # Potential Energy
#cmap = np.copy(COLVAR[:, 2])   # Collective Variable (Q)
bias = np.copy(COLVAR[:, 5])   # Bias Potential
#rg = np.copy(COLVAR[:, 7]) 

phi_AF= -102.417
psi_AF= -1.44349
print (len(phi), len (time), time [-1], time[0], time[1]) 
kb=0.00831446 # kJ/mol/K
temperature=390 # K
beta=1./(kb*temperature)
newTemperature=300 # K
newBeta=1./(kb*newTemperature)
logweights=beta*bias+(beta-newBeta)*energy
logweights -= np.amax(logweights)
weights=np.exp(logweights)


H1, xedges1, yedges1 = np.histogram2d(phi, psi, weights=weights,bins=(100,100),range=[[-200,200],[-200,200]])
H1 = H1.T
H1=-(1/newBeta)*np.log(gaussian_filter(H1, sigma=3))
H1 -= np.amin(H1) 
X1, Y1 = np.meshgrid(xedges1, yedges1)
cmap = plt.cm.hot
plt.pcolormesh(X1, Y1, H1, vmin=0,vmax=15, cmap=cmap)
plt.colorbar()
x = (X1[1:,1:]+X1[:-1,:-1])/2.
y = (Y1[1:,1:]+Y1[:-1,:-1])/2.
plt.contour(x,y, H1 ,np.arange(0,25,5),colors='black',linewidths=0.5)

plt.xlabel("Phi (ϕ) Angle (degrees)", fontsize=20)
plt.ylabel("Psi (ψ) Angle (degrees)",fontsize=20)
plt.tick_params(axis='x', labelsize=20)
plt.tick_params(axis='y', labelsize=20)
plt.scatter(phi_AF, psi_AF, color='blue')

plt.show()

In [ ]:
ignore = 25000
num_replicas = 16

# --- 1. Load and Process RAMA Data ---
# Load the full concatenated file
rama_file = '/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/rama/rama_turn1-ASN-121.xvg'
with open(rama_file, 'r') as f:
    raw_rama = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)

# Calculate frames per replica to verify consistency
total_rama_frames = raw_rama.shape[0]
frames_per_replica = total_rama_frames // num_replicas

if total_rama_frames % num_replicas != 0:
    print(f"Warning: Total rama frames ({total_rama_frames}) is not divisible by {num_replicas}!")

# Reshape into (16, frames_per_replica, columns)
rama_reshaped = raw_rama.reshape((num_replicas, frames_per_replica, raw_rama.shape[1]))

# Slice off the 'ignore' frames from the second dimension (axis 1)
rama_sliced = rama_reshaped[:, ignore:, :]

# Flatten back to 2D array: (Total_Effective_Frames, columns)
rama_final = rama_sliced.reshape((-1, raw_rama.shape[1]))

# Extract Phi and Psi
phi = rama_final[:, 0]
psi = rama_final[:, 1]

COLVAR1 = np.genfromtxt("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/0/pasted_2000ns")[ignore:, :]
print(f"Replica 0 loaded: {len(COLVAR1)} frames")

for i in range(1, 16):  # Load data for replicas 1 to 15
    file_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/pasted_2000ns"
    # Concatenate subsequent files
    COLVAR1 = np.concatenate((COLVAR1, np.genfromtxt(file_path)[ignore:, :]))
    # print(f"Replica {i} loaded")

COLVAR = COLVAR1
time = np.copy(COLVAR[:, 0])
energy = np.copy(COLVAR[:, 1]) # Potential Energy
#cmap = np.copy(COLVAR[:, 2])   # Collective Variable (Q)
bias = np.copy(COLVAR[:, 5])   # Bias Potential
#rg = np.copy(COLVAR[:, 7]) 

phi_AF= -111.163
psi_AF= 33.6202
print (len(phi), len (time), time [-1], time[0], time[1]) 
kb=0.00831446 # kJ/mol/K
temperature=390 # K
beta=1./(kb*temperature)
newTemperature=300 # K
newBeta=1./(kb*newTemperature)
logweights=beta*bias+(beta-newBeta)*energy
logweights -= np.amax(logweights)
weights=np.exp(logweights)


H1, xedges1, yedges1 = np.histogram2d(phi, psi, weights=weights,bins=(100,100),range=[[-200,200],[-200,200]])
H1 = H1.T
H1=-(1/newBeta)*np.log(gaussian_filter(H1, sigma=3))
H1 -= np.amin(H1) 
X1, Y1 = np.meshgrid(xedges1, yedges1)
cmap = plt.cm.hot
plt.pcolormesh(X1, Y1, H1, vmin=0,vmax=15, cmap=cmap)
plt.colorbar()
x = (X1[1:,1:]+X1[:-1,:-1])/2.
y = (Y1[1:,1:]+Y1[:-1,:-1])/2.
plt.contour(x,y, H1 ,np.arange(0,25,5),colors='black',linewidths=0.5)

plt.xlabel("Phi (ϕ) Angle (degrees)", fontsize=20)
plt.ylabel("Psi (ψ) Angle (degrees)",fontsize=20)
plt.tick_params(axis='x', labelsize=20)
plt.tick_params(axis='y', labelsize=20)
plt.scatter(phi_AF, psi_AF, color='blue')

plt.show()

In [ ]:
ignore = 25000
num_replicas = 16

# --- 1. Load and Process RAMA Data ---
# Load the full concatenated file
rama_file = '/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/rama/rama_turn1_PRO-120.xvg'
with open(rama_file, 'r') as f:
    raw_rama = np.genfromtxt((line for line in f if not line.startswith(('#', '@'))), dtype=float)

# Calculate frames per replica to verify consistency
total_rama_frames = raw_rama.shape[0]
frames_per_replica = total_rama_frames // num_replicas

if total_rama_frames % num_replicas != 0:
    print(f"Warning: Total rama frames ({total_rama_frames}) is not divisible by {num_replicas}!")

# Reshape into (16, frames_per_replica, columns)
rama_reshaped = raw_rama.reshape((num_replicas, frames_per_replica, raw_rama.shape[1]))

# Slice off the 'ignore' frames from the second dimension (axis 1)
rama_sliced = rama_reshaped[:, ignore:, :]

# Flatten back to 2D array: (Total_Effective_Frames, columns)
rama_final = rama_sliced.reshape((-1, raw_rama.shape[1]))

# Extract Phi and Psi
phi = rama_final[:, 0]
psi = rama_final[:, 1]

COLVAR1 = np.genfromtxt("/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/0/pasted_2000ns")[ignore:, :]
print(f"Replica 0 loaded: {len(COLVAR1)} frames")

for i in range(1, 16):  # Load data for replicas 1 to 15
    file_path = f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/pasted_2000ns"
    # Concatenate subsequent files
    COLVAR1 = np.concatenate((COLVAR1, np.genfromtxt(file_path)[ignore:, :]))
    # print(f"Replica {i} loaded")

COLVAR = COLVAR1
time = np.copy(COLVAR[:, 0])
energy = np.copy(COLVAR[:, 1]) # Potential Energy
#cmap = np.copy(COLVAR[:, 2])   # Collective Variable (Q)
bias = np.copy(COLVAR[:, 5])   # Bias Potential
#rg = np.copy(COLVAR[:, 7]) 

phi_AF= -80.2232
psi_AF= -19.5668
print (len(phi), len (time), time [-1], time[0], time[1]) 
kb=0.00831446 # kJ/mol/K
temperature=390 # K
beta=1./(kb*temperature)
newTemperature=300 # K
newBeta=1./(kb*newTemperature)
logweights=beta*bias+(beta-newBeta)*energy
logweights -= np.amax(logweights)
weights=np.exp(logweights)


H1, xedges1, yedges1 = np.histogram2d(phi, psi, weights=weights,bins=(100,100),range=[[-200,200],[-200,200]])
H1 = H1.T
H1=-(1/newBeta)*np.log(gaussian_filter(H1, sigma=3))
H1 -= np.amin(H1) 
X1, Y1 = np.meshgrid(xedges1, yedges1)
cmap = plt.cm.hot
plt.pcolormesh(X1, Y1, H1, vmin=0,vmax=15, cmap=cmap)
plt.colorbar()
x = (X1[1:,1:]+X1[:-1,:-1])/2.
y = (Y1[1:,1:]+Y1[:-1,:-1])/2.
plt.contour(x,y, H1 ,np.arange(0,25,5),colors='black',linewidths=0.5)

plt.xlabel("Phi (ϕ) Angle (degrees)", fontsize=20)
plt.ylabel("Psi (ψ) Angle (degrees)",fontsize=20)
plt.tick_params(axis='x', labelsize=20)
plt.tick_params(axis='y', labelsize=20)
plt.scatter(phi_AF, psi_AF, color='blue')

plt.show()

In [ ]:
# File path template
file_template = "/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/dssp/proteina_charmm36_opes_all_nd{i}_prot.xpm"

# Initialize a list to hold the matrices
reside_ss_list = []

# Loop over indices from 0 to 20
for i in range(16):
    # Generate the file path for the current index
    xpm_file_path = file_template.format(i=i)
    
    # Open the file and process it
    with open(xpm_file_path, 'r') as file:
        lines = file.readlines()
    
    # Start from line 919 (Python's zero-based index)
  #  if i == 4:
   #     matrix_start = 1287
    #else:
        matrix_start = 1288
    
    matrix_lines = lines[matrix_start:]

    # Parse the matrix
    matrix = []
    for line in matrix_lines:
        # Remove extra quotes, commas, and whitespac
        clean_line = line.strip().strip(',').strip('"')
        # Skip empty lines
        if not clean_line:
            continue
        # Convert characters to a list and append to the matrix
        matrix.append(list(clean_line))
    
    # Convert to numpy array
    reside_ss = np.array(matrix)
    
    # Store the matrix in the list
    reside_ss_list.append(reside_ss)

# Unpack the matrices into individual variables if needed
(
    reside_ss0, reside_ss1, reside_ss2, reside_ss3, reside_ss4,
    reside_ss5, reside_ss6, reside_ss7, reside_ss8, reside_ss9,
    reside_ss10, reside_ss11, reside_ss12, reside_ss13, reside_ss14,reside_ss15
) = reside_ss_list

# Define the number of columns (frames) to ignore
ignore_columns = 25000  # Ignore this many columns from the beginning

# Concatenate and process data for residues across all files
# Since the rows are residues and columns are time, we slice columns to ignore the initial frames
concatenated_residues = []
num_residues = len(reside_ss_list[0])
print(num_residues)

# For each residue (row index) in the first file (assuming all files have the same structure)
for residue_index in range(num_residues):
    # Extract the row for this residue from each matrix, ignoring the initial frames
    concatenated_row = np.concatenate([
        reside_ss[residue_index, ignore_columns:]
        for reside_ss in reside_ss_list
    ])
    # Append the concatenated row to the result
    concatenated_residues.append(concatenated_row)

# Convert to a numpy array and transpose the matrix
final_output_a99sb = np.array(concatenated_residues).T

# The resulting `final_output_a99sb` has rows = time points (after ignoring frames) 
# and columns = residues

# Print the shape for verification
print("Shape of final_output:", final_output_a99sb.shape)

# Concatenate COLVAR data
COLVAR = np.concatenate([np.genfromtxt(
    f"/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/hills/{i}/pasted_2000ns"
)[25000:, :] for i in range(16)])

# Extract relevant columns from COLVAR
time_colvar = np.copy(COLVAR[:, 0])  # Time
energy = np.copy(COLVAR[:, 1])       # Energy
bias   = np.copy(COLVAR[:, 5])       # Bias

# Constants
kb = 0.00831446  # kJ/mol/K
temperature = 390  # K
new_temperature = 300  # K
beta = 1.0 / (kb * temperature)
new_beta = 1.0 / (kb * new_temperature)

# Generate logweights from bias
logweights = beta * bias + (beta - new_beta) * energy
logweights -= np.amax(logweights)
weights = np.exp(logweights)  # Weights for histogram

# Initialize arrays to store propensities for each secondary structure type
num_residues = final_output_a99sb.shape[1]
print(num_residues)
propensities_beta_sheet  = np.zeros(num_residues)
propensities_alpha_helix = np.zeros(num_residues)
propensities_coil        = np.zeros(num_residues)

# Secondary structure mappings
structure_map = {
    "E": "beta_sheet", "B": "beta_sheet",
    "H": "helix", "I": "helix", "G": "helix",
    "~": "coil", "T": "coil", "S": "coil"
}

# Count occurrences weighted by logweights for each residue
for residue_index in range(num_residues):
    # Extract secondary structure of the current residue across all time frames
    residue_structure = final_output_a99sb[:, residue_index]  # secondary structure for each residue
    
    # Initialize counts for each secondary structure type
    weighted_counts = {"beta_sheet": 0.0, "helix": 0.0, "coil": 0.0}
    total_weight = 0.0

    # Calculate weighted counts
    for time_index, structure in enumerate(residue_structure):
        if structure in structure_map:
            weighted_counts[structure_map[structure]] += weights[time_index]
        total_weight += weights[time_index]

    # Normalize counts to propensities (limited to 0 to 1)
    if total_weight > 0:
        propensities_beta_sheet[residue_index]  = weighted_counts["beta_sheet"] / total_weight
        propensities_alpha_helix[residue_index] = weighted_counts["helix"]      / total_weight
        propensities_coil[residue_index]        = weighted_counts["coil"]       / total_weight

# ==========================
# PLOTTING (modified block)
# ==========================

# Residue index array (1..num_residues)
residue_numbers = np.arange(1, num_residues + 1)

# Map to the variable names used in your plotting template
res_idx   = residue_numbers
prop_beta = propensities_beta_sheet
prop_helix = propensities_alpha_helix
prop_coil  = propensities_coil
n_res      = num_residues
prop_beta = np.flip(prop_beta)
prop_helix = np.flip(prop_helix)
prop_coil = np.flip (prop_coil)
# Colors
coil_color  = "#d9d9d9"  # light gray
beta_color  = "#ffd300"  # strong yellow
helix_color = "#cc33cc"  # purple-pink (VMD-like)

plt.figure(figsize=(14, 8))

plt.plot(res_idx, prop_beta,
         label="β-sheet",
         linewidth=3.0,
         color=beta_color,
         linestyle='--',
         marker='o')

plt.plot(res_idx, prop_helix,
         label="Helix",
         linewidth=3.0,
         color=helix_color,
         linestyle='--',
         marker='o')

# If you later want to show coil, just uncomment:
# plt.plot(res_idx, prop_coil,
#          label="Coil",
#          linewidth=2.0,
#          color=coil_color)

plt.xlabel("Residue ID")
plt.ylabel("Propensity")
n_res = num_residues
plt.xlim(1, num_residues)
plt.ylim(0, 1)
#OFFSET = 0
#tick_positions = np.arange(1, n_res + 1, 12)
#tick_labels    = tick_positions + OFFSET
# These assume you already defined tick_positions, tick_labels earlier
plt.xticks(np.arange(1, num_residues+1, 10), fontsize=35, rotation=90)
plt.yticks(fontsize=35)
plt.tight_layout(rect=[0, 0, 0.82, 1])

#out_png = "dssp_rrm-charmm_weighted.png"
#plt.savefig(out_png, dpi=400)
plt.show()



In [ ]:
plt.figure(figsize=(14, 8))

plt.plot(res_idx, prop_beta,
         label="β-sheet",
         linewidth=3.0,
         color=beta_color,
         linestyle='--',
         marker='o')

plt.plot(res_idx, prop_helix,
         label="Helix",
         linewidth=3.0,
         color=helix_color,
         linestyle='--',
         marker='o')

# If you later want to show coil, just uncomment:
# plt.plot(res_idx, prop_coil,
#          label="Coil",
#          linewidth=2.0,
#          color=coil_color)

plt.xlabel("Residue ID")
plt.ylabel("Propensity")
n_res = num_residues
plt.xlim(1, num_residues)
plt.ylim(0, 1)
#OFFSET = 0
#tick_positions = np.arange(1, n_res + 1, 12)
#tick_labels    = tick_positions + OFFSET
# These assume you already defined tick_positions, tick_labels earlier
plt.xticks(np.arange(1, num_residues+1, 10), fontsize=35, rotation=90)
plt.yticks(fontsize=35)
plt.tight_layout(rect=[0, 0, 0.82, 1])

#out_png = "dssp_rrm-charmm_weighted.png"
#plt.savefig(out_png, dpi=400)
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

# ===== INPUT =====
xpm_path = Path("/project/zerze/krahimin/proteinA_all/2B/round3/data/dssp/em_prot.xpm")

START_LINE_1BASED = 18
START_IDX = START_LINE_1BASED - 1

# ===== Parse DSSP XPM =====
def load_dssp_matrix_from_line(path, start_idx):
    """
    Reads an XPM file and parses rows of quoted characters beginning at start_idx.
    Stops when it encounters a line without quotes or a closing brace.
    Returns a 2D numpy array of shape (n_residues, n_frames) with single-char DSSP codes.
    """
    with open(path, "r") as f:
        lines = f.readlines()

    rows = []
    for line in lines[start_idx:]:
        if "}" in line:
            break
        # Expect data lines like: "HHH~TT..." ,
        if '"' not in line:
            # Non-data line encountered after data started → stop
            if rows:
                break
            else:
                continue
        payload = line.split('"', 2)[1]  # content between first pair of quotes
        payload = payload.rstrip().rstrip(",")
        if payload == "":
            if rows:
                break
            else:
                continue
        rows.append(list(payload))

    if not rows:
        raise ValueError("No DSSP data parsed. Check START_LINE or file format.")

    # rows: residues; columns: frames
    return np.array(rows, dtype="<U1")

residue_by_time = load_dssp_matrix_from_line(xpm_path, START_IDX)
n_res, n_frames = residue_by_time.shape
print(f"Parsed from line {START_LINE_1BASED}: residues={n_res}, frames={n_frames}")


# ===== Convert DSSP to 3-state =====
to_class = {
    "E": "beta", "B": "beta",
    "H": "helix", "I": "helix", "G": "helix",
    "~": "coil", "T": "coil", "S": "coil", " ": "coil", ".": "coil"
}

prop_beta  = np.zeros(n_res, dtype=float)
prop_helix = np.zeros(n_res, dtype=float)
prop_coil  = np.zeros(n_res, dtype=float)

for i in range(n_res):
    row = residue_by_time[i, :]
    beta  = np.sum([to_class.get(ch, "coil") == "beta"  for ch in row])
    helix = np.sum([to_class.get(ch, "coil") == "helix" for ch in row])
    coil  = len(row) - beta - helix
    total = len(row)
    if total > 0:
        prop_beta[i]  = beta  / total
        prop_helix[i] = helix / total
        prop_coil[i]  = coil  / total

# If you need to flip residue order to match a prior orientation, toggle this:
# prop_beta  = prop_beta[::-1]
# prop_helix = prop_helix[::-1]
# prop_coil  = prop_coil[::-1]
prop_beta = np.flip(prop_beta)
prop_helix = np.flip(prop_helix)
prop_coil = np.flip (prop_coil)
# Only one frame expected
frame = residue_by_time[:, 0]

structure_classes = [to_class.get(ch, "coil") for ch in frame]
res_indices = np.arange(1, n_res + 1)
beta_mask = prop_beta == 1.0
helix_mask = prop_helix == 1.0

# ===== Colors =====

# Requested colors:
coil_color = "#d9d9d9"  # light gray
beta_color = "#ffd300"  # strong yellow
helix_color = "#cc33cc" # purple-pink (VMD-like)

# ===== Plot =====
#res_idx = np.arange(RES_START, RES_END + 1)

plt.figure(figsize=(14, 2))
if np.any(beta_mask):
    plt.scatter(res_indices[beta_mask], np.ones(beta_mask.sum()), color=beta_color, s=60, label="β-sheet")
if np.any(helix_mask):
    plt.scatter(res_indices[helix_mask], np.ones(helix_mask.sum()), color=helix_color, s=60, label="Helix")

# Legend outside
# Put legend outside (right)
#plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0.)
#plt.tight_layout(rect=[0, 0, 0.82, 1])

plt.xlabel("Residue ID")


plt.yticks( fontsize=30)
#plt.tight_layout(rect=[0, 0, 0.82, 1])
plt.xticks(np.arange(1, num_residues+1, 10), fontsize=35, rotation=90)


plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

# ===== INPUT =====
xpm_path = Path("/project/zerze/krahimin/proteinA_all/2B/round3/data/dssp/4NPF-2B.xpm")

START_LINE_1BASED = 18
START_IDX = START_LINE_1BASED - 1

# ===== Parse DSSP XPM =====
def load_dssp_matrix_from_line(path, start_idx):
    """
    Reads an XPM file and parses rows of quoted characters beginning at start_idx.
    Stops when it encounters a line without quotes or a closing brace.
    Returns a 2D numpy array of shape (n_residues, n_frames) with single-char DSSP codes.
    """
    with open(path, "r") as f:
        lines = f.readlines()

    rows = []
    for line in lines[start_idx:]:
        if "}" in line:
            break
        # Expect data lines like: "HHH~TT..." ,
        if '"' not in line:
            # Non-data line encountered after data started → stop
            if rows:
                break
            else:
                continue
        payload = line.split('"', 2)[1]  # content between first pair of quotes
        payload = payload.rstrip().rstrip(",")
        if payload == "":
            if rows:
                break
            else:
                continue
        rows.append(list(payload))

    if not rows:
        raise ValueError("No DSSP data parsed. Check START_LINE or file format.")

    # rows: residues; columns: frames
    return np.array(rows, dtype="<U1")

residue_by_time = load_dssp_matrix_from_line(xpm_path, START_IDX)
n_res, n_frames = residue_by_time.shape
print(f"Parsed from line {START_LINE_1BASED}: residues={n_res}, frames={n_frames}")


# ===== Convert DSSP to 3-state =====
to_class = {
    "E": "beta", "B": "beta",
    "H": "helix", "I": "helix", "G": "helix",
    "~": "coil", "T": "coil", "S": "coil", " ": "coil", ".": "coil"
}

prop_beta  = np.zeros(n_res, dtype=float)
prop_helix = np.zeros(n_res, dtype=float)
prop_coil  = np.zeros(n_res, dtype=float)

for i in range(n_res):
    row = residue_by_time[i, :]
    beta  = np.sum([to_class.get(ch, "coil") == "beta"  for ch in row])
    helix = np.sum([to_class.get(ch, "coil") == "helix" for ch in row])
    coil  = len(row) - beta - helix
    total = len(row)
    if total > 0:
        prop_beta[i]  = beta  / total
        prop_helix[i] = helix / total
        prop_coil[i]  = coil  / total

# If you need to flip residue order to match a prior orientation, toggle this:
# prop_beta  = prop_beta[::-1]
# prop_helix = prop_helix[::-1]
# prop_coil  = prop_coil[::-1]
prop_beta = np.flip(prop_beta)
prop_helix = np.flip(prop_helix)
prop_coil = np.flip (prop_coil)
# Only one frame expected
frame = residue_by_time[:, 0]

structure_classes = [to_class.get(ch, "coil") for ch in frame]
res_indices = np.arange(1, n_res + 1)
beta_mask = prop_beta == 1.0
helix_mask = prop_helix == 1.0

# ===== Colors =====

# Requested colors:
coil_color = "#d9d9d9"  # light gray
beta_color = "#ffd300"  # strong yellow
helix_color = "#cc33cc" # purple-pink (VMD-like)

# ===== Plot =====
#res_idx = np.arange(RES_START, RES_END + 1)

plt.figure(figsize=(14, 2))
if np.any(beta_mask):
    plt.scatter(res_indices[beta_mask], np.ones(beta_mask.sum()), color=beta_color, s=60, label="β-sheet")
if np.any(helix_mask):
    plt.scatter(res_indices[helix_mask], np.ones(helix_mask.sum()), color=helix_color, s=60, label="Helix")

# Legend outside
# Put legend outside (right)
#plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0.)
#plt.tight_layout(rect=[0, 0, 0.82, 1])

plt.xlabel("Residue ID")


plt.yticks( fontsize=30)
#plt.tight_layout(rect=[0, 0, 0.82, 1])
plt.xticks(np.arange(1, num_residues+1, 10), fontsize=35, rotation=90)


plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Configuration ---
xpm_file_path = "/project/zerze/krahimin/proteinA_all/2B/round3/data/dssp/cltr_dmn2_pdb0001.xpm"
start_line_index = 78   # Python index: this skips lines 0-77 and starts reading at line 78
ignore_frames = 0       # Set this if you need to ignore the first N frames of the actual data

# --- Data Loading Section ---
print(f"Reading file: {xpm_file_path}")

with open(xpm_file_path, 'r') as file:
    lines = file.readlines()

# Slice the lines starting from the user-specified index
# Ensure we don't go out of bounds if the file is shorter
if len(lines) > start_line_index:
    matrix_lines = lines[start_line_index:]
else:
    raise ValueError(f"File has fewer lines ({len(lines)}) than the start index ({start_line_index}).")

matrix = []
for line in matrix_lines:
    # Remove extra quotes, commas, and whitespace typical of XPM format
    clean_line = line.strip().strip(',').strip('"')
    
    # Skip empty lines that might appear at the end
    if not clean_line:
        continue
        
    # Convert string to list of characters
    matrix.append(list(clean_line))

# Convert to numpy array
# XPM shape is usually [Residues, Time] (Rows=Residues)
raw_data = np.array(matrix)

print(f"Raw data shape loaded: {raw_data.shape}")

# Apply Ignore frames (slicing columns)
# We assume raw_data is [Residues, Time]. 
final_data = raw_data[:, ignore_frames:]

# Transpose to get [Time, Residues] for easier looping
# Final shape: (N_frames, N_residues)
ss_data = final_data.T 

print(f"Processed shape (Frames, Residues): {ss_data.shape}")
num_residues = ss_data.shape[1]
num_frames = ss_data.shape[0]

# --- Unweighted Calculation Section ---
propensities_beta_sheet  = np.zeros(num_residues)
propensities_alpha_helix = np.zeros(num_residues)
propensities_coil        = np.zeros(num_residues)

# DSSP Mapping
structure_map = {
    "E": "beta_sheet", "B": "beta_sheet",
    "H": "helix", "I": "helix", "G": "helix",
    "~": "coil", "T": "coil", "S": "coil", 
    " ": "coil"
}

print("Calculating unweighted propensities...")

for r in range(num_residues):
    # Get the trajectory for this specific residue
    residue_traj = ss_data[:, r]
    
    counts = {"beta_sheet": 0, "helix": 0, "coil": 0}
    
    for structure in residue_traj:
        # Map character to generic structure type (default to coil)
        stype = structure_map.get(structure, "coil") 
        counts[stype] += 1
        
    # Calculate Frequency (Count / Total Frames)
    if num_frames > 0:
        propensities_beta_sheet[r]  = counts["beta_sheet"] / num_frames
        propensities_alpha_helix[r] = counts["helix"]      / num_frames
        propensities_coil[r]        = counts["coil"]       / num_frames

# --- Plotting Section ---
residue_numbers = np.arange(1, num_residues + 1)

# Apply flip (From your previous code preferences)
# Remove these 3 lines if residue 1 appears at the wrong end of the x-axis
prop_beta = np.flip(propensities_beta_sheet)
prop_helix = np.flip(propensities_alpha_helix)
prop_coil = np.flip(propensities_coil)

# Colors
beta_color  = "#ffd300"  # strong yellow
helix_color = "#cc33cc"  # purple-pink

plt.figure(figsize=(14, 8))

# Plot Beta Sheet
plt.plot(residue_numbers, prop_beta,
         label="β-sheet",
         linewidth=3.0,
         color=beta_color,
         linestyle='--',
         marker='o')

# Plot Helix
plt.plot(residue_numbers, prop_helix,
         label="Helix",
         linewidth=3.0,
         color=helix_color,
         linestyle='--',
         marker='o')

plt.xlabel("Residue ID", fontsize=30)
plt.ylabel("Propensity", fontsize=30)

# Axis limits
plt.xlim(1, num_residues)
plt.ylim(0, 1)

# Ticks configuration
tick_positions = np.arange(1, num_residues + 1, 10) 
plt.xticks(tick_positions, tick_positions, fontsize=30, rotation=90)
plt.yticks(np.arange(0, 1.1, 0.2), fontsize=30)

plt.tight_layout(rect=[0, 0, 0.95, 1])
# plt.legend(fontsize=20) 
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Configuration ---
xpm_file_path = "/project/zerze/krahimin/proteinA_all/2B/round3/data/dssp/proteina_charmm36_opes_all_nd0_prot.xpm"
start_line_index = 1288   # Python index: this skips lines 0-77 and starts reading at line 78
ignore_frames = 0       # Set this if you need to ignore the first N frames of the actual data

# --- Data Loading Section ---
print(f"Reading file: {xpm_file_path}")

with open(xpm_file_path, 'r') as file:
    lines = file.readlines()

# Slice the lines starting from the user-specified index
# Ensure we don't go out of bounds if the file is shorter
if len(lines) > start_line_index:
    matrix_lines = lines[start_line_index:]
else:
    raise ValueError(f"File has fewer lines ({len(lines)}) than the start index ({start_line_index}).")

matrix = []
for line in matrix_lines:
    # Remove extra quotes, commas, and whitespace typical of XPM format
    clean_line = line.strip().strip(',').strip('"')
    
    # Skip empty lines that might appear at the end
    if not clean_line:
        continue
        
    # Convert string to list of characters
    matrix.append(list(clean_line))

# Convert to numpy array
# XPM shape is usually [Residues, Time] (Rows=Residues)
raw_data = np.array(matrix)

print(f"Raw data shape loaded: {raw_data.shape}")

# Apply Ignore frames (slicing columns)
# We assume raw_data is [Residues, Time]. 
final_data = raw_data[:, ignore_frames:]

# Transpose to get [Time, Residues] for easier looping
# Final shape: (N_frames, N_residues)
ss_data = final_data.T 

print(f"Processed shape (Frames, Residues): {ss_data.shape}")
num_residues = ss_data.shape[1]
num_frames = ss_data.shape[0]

# --- Unweighted Calculation Section ---
propensities_beta_sheet  = np.zeros(num_residues)
propensities_alpha_helix = np.zeros(num_residues)
propensities_coil        = np.zeros(num_residues)

# DSSP Mapping
structure_map = {
    "E": "beta_sheet", "B": "beta_sheet",
    "H": "helix", "I": "helix", "G": "helix",
    "~": "coil", "T": "coil", "S": "coil", 
    " ": "coil"
}

print("Calculating unweighted propensities...")

for r in range(num_residues):
    # Get the trajectory for this specific residue
    residue_traj = ss_data[:, r]
    
    counts = {"beta_sheet": 0, "helix": 0, "coil": 0}
    
    for structure in residue_traj:
        # Map character to generic structure type (default to coil)
        stype = structure_map.get(structure, "coil") 
        counts[stype] += 1
        
    # Calculate Frequency (Count / Total Frames)
    if num_frames > 0:
        propensities_beta_sheet[r]  = counts["beta_sheet"] / num_frames
        propensities_alpha_helix[r] = counts["helix"]      / num_frames
        propensities_coil[r]        = counts["coil"]       / num_frames

# --- Plotting Section ---
residue_numbers = np.arange(1, num_residues + 1)

# Apply flip (From your previous code preferences)
# Remove these 3 lines if residue 1 appears at the wrong end of the x-axis
prop_beta = np.flip(propensities_beta_sheet)
prop_helix = np.flip(propensities_alpha_helix)
prop_coil = np.flip(propensities_coil)

# Colors
beta_color  = "#ffd300"  # strong yellow
helix_color = "#cc33cc"  # purple-pink

plt.figure(figsize=(14, 8))

# Plot Beta Sheet
plt.plot(residue_numbers, prop_beta,
         label="β-sheet",
         linewidth=3.0,
         color=beta_color,
         linestyle='--',
         marker='o')

# Plot Helix
plt.plot(residue_numbers, prop_helix,
         label="Helix",
         linewidth=3.0,
         color=helix_color,
         linestyle='--',
         marker='o')

plt.xlabel("Residue ID", fontsize=30)
plt.ylabel("Propensity", fontsize=30)

# Axis limits
plt.xlim(1, num_residues)
plt.ylim(0, 1)

y_values=[6,18,24,37,42,55,64,76,82,95,100,113]
for y in y_values:
    plt.axvline(x=y, color='grey', linestyle='--', linewidth=1)
    
# Ticks configuration
tick_positions = np.arange(1, num_residues + 1, 10) 
plt.xticks(tick_positions, tick_positions, fontsize=30, rotation=90)
plt.yticks(np.arange(0, 1.1, 0.2), fontsize=30)

plt.tight_layout(rect=[0, 0, 0.95, 1])
# plt.legend(fontsize=20) 
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Configuration ---
xpm_file_path = "/project/zerze/krahimin/proteinA_all/2B/round3/data/dssp/proteina_charmm36_opes_all_nd15_prot.xpm"
start_line_index = 1288   # Python index: this skips lines 0-77 and starts reading at line 78
ignore_frames = 0       # Set this if you need to ignore the first N frames of the actual data

# --- Data Loading Section ---
print(f"Reading file: {xpm_file_path}")

with open(xpm_file_path, 'r') as file:
    lines = file.readlines()

# Slice the lines starting from the user-specified index
# Ensure we don't go out of bounds if the file is shorter
if len(lines) > start_line_index:
    matrix_lines = lines[start_line_index:]
else:
    raise ValueError(f"File has fewer lines ({len(lines)}) than the start index ({start_line_index}).")

matrix = []
for line in matrix_lines:
    # Remove extra quotes, commas, and whitespace typical of XPM format
    clean_line = line.strip().strip(',').strip('"')
    
    # Skip empty lines that might appear at the end
    if not clean_line:
        continue
        
    # Convert string to list of characters
    matrix.append(list(clean_line))

# Convert to numpy array
# XPM shape is usually [Residues, Time] (Rows=Residues)
raw_data = np.array(matrix)

print(f"Raw data shape loaded: {raw_data.shape}")

# Apply Ignore frames (slicing columns)
# We assume raw_data is [Residues, Time]. 
final_data = raw_data[:, ignore_frames:]

# Transpose to get [Time, Residues] for easier looping
# Final shape: (N_frames, N_residues)
ss_data = final_data.T 

print(f"Processed shape (Frames, Residues): {ss_data.shape}")
num_residues = ss_data.shape[1]
num_frames = ss_data.shape[0]

# --- Unweighted Calculation Section ---
propensities_beta_sheet  = np.zeros(num_residues)
propensities_alpha_helix = np.zeros(num_residues)
propensities_coil        = np.zeros(num_residues)

# DSSP Mapping
structure_map = {
    "E": "beta_sheet", "B": "beta_sheet",
    "H": "helix", "I": "helix", "G": "helix",
    "~": "coil", "T": "coil", "S": "coil", 
    " ": "coil"
}

print("Calculating unweighted propensities...")

for r in range(num_residues):
    # Get the trajectory for this specific residue
    residue_traj = ss_data[:, r]
    
    counts = {"beta_sheet": 0, "helix": 0, "coil": 0}
    
    for structure in residue_traj:
        # Map character to generic structure type (default to coil)
        stype = structure_map.get(structure, "coil") 
        counts[stype] += 1
        
    # Calculate Frequency (Count / Total Frames)
    if num_frames > 0:
        propensities_beta_sheet[r]  = counts["beta_sheet"] / num_frames
        propensities_alpha_helix[r] = counts["helix"]      / num_frames
        propensities_coil[r]        = counts["coil"]       / num_frames

# --- Plotting Section ---
residue_numbers = np.arange(1, num_residues + 1)

# Apply flip (From your previous code preferences)
# Remove these 3 lines if residue 1 appears at the wrong end of the x-axis
prop_beta = np.flip(propensities_beta_sheet)
prop_helix = np.flip(propensities_alpha_helix)
prop_coil = np.flip(propensities_coil)

# Colors
beta_color  = "#ffd300"  # strong yellow
helix_color = "#cc33cc"  # purple-pink

plt.figure(figsize=(14, 8))

# Plot Beta Sheet
plt.plot(residue_numbers, prop_beta,
         label="β-sheet",
         linewidth=3.0,
         color=beta_color,
         linestyle='--',
         marker='o')

# Plot Helix
plt.plot(residue_numbers, prop_helix,
         label="Helix",
         linewidth=3.0,
         color=helix_color,
         linestyle='--',
         marker='o')

plt.xlabel("Residue ID", fontsize=30)
plt.ylabel("Propensity", fontsize=30)

# Axis limits
plt.xlim(1, num_residues)
plt.ylim(0, 1)

y_values=[6,18,24,37,42,55,64,76,82,95,100,113]
for y in y_values:
    plt.axvline(x=y, color='grey', linestyle='--', linewidth=1)
    
# Ticks configuration
tick_positions = np.arange(1, num_residues + 1, 10) 
plt.xticks(tick_positions, tick_positions, fontsize=30, rotation=90)
plt.yticks(np.arange(0, 1.1, 0.2), fontsize=30)

plt.tight_layout(rect=[0, 0, 0.95, 1])
# plt.legend(fontsize=20) 
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Define the directories for each system as provided
# Added the final slash to Natural ED for consistency, though os.path.join handles it
systems_paths = {
    "Natural 2B": "/project/zerze/krahimin/proteinA_all/2B/round3/data/rmsf/",
    "2B-Mutated linker": "/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/rmsf/",
    "Natural ED": "/project/zerze/krahimin/proteinA_all/ED_round3/data/rmsf/"
}

# List of segments corresponding to your file naming convention
segments = ["all", "linker", "D1H1", "D1H2", "D1H3", "D2H1", "D2H2", "D2H3"]
colors = ['black', 'red', 'green']
def get_average_rmsf(filepath):
    """
    Parses a GROMACS .xvg file and returns the mean of the RMSF values.
    GROMACS .xvg files typically have atom/residue index in col 0 and RMSF in col 1.
    """
    try:
        # Load numerical data, ignoring comments and metadata lines
        data = np.loadtxt(filepath, comments=['#', '@'])
        # Return the average of the second column (RMSF in nm)
        return np.mean(data[:, 1])
    except FileNotFoundError:
        print(f"Warning: File not found: {filepath}")
        return 0.0
    except Exception as e:
        print(f"Error processing {filepath}: {e}")
        return 0.0

# Dictionary to store the calculated averages
results = {name: [] for name in systems_paths.keys()}

# Data Processing
for system_name, base_path in systems_paths.items():
    for seg in segments:
        file_name = f"rmsf_{seg}.xvg"
        full_path = os.path.join(base_path, file_name)
        avg_rmsf = get_average_rmsf(full_path)
        results[system_name].append(avg_rmsf)

# Visualization
x = np.arange(len(segments))  # Label locations
width = 0.25                  # Width of the bars

fig, ax = plt.subplots(figsize=(12, 6), dpi=100)

# Plotting each system
for i, (system_name, averages) in enumerate(results.items()):
    # Offset each system's bars to create a grouped effect
    offset = (i - 1) * width 
    ax.bar(x + offset, averages, width, color=colors[i],label=system_name, edgecolor='black', alpha=0.8)

# Formatting the plot according to scientific standards
ax.set_ylabel('Average RMSF (nm)', fontsize=20, fontweight='bold')
#ax.set_title('Regional Average RMSF Comparison across Systems', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(segments, fontsize=20)
ax.legend(frameon=False)
ax.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('rmsf_averages_comparison.png')
plt.show()

# Print numerical summary for verification
print(f"{'Segment':<10} | {'Natural 2B':<12} | {'Mutated':<12} | {'Natural ED':<12}")
print("-" * 55)
for idx, seg in enumerate(segments):
    val1 = results["Natural 2B"][idx]
    val2 = results["2B-Mutated linker"][idx]
    val3 = results["Natural ED"][idx]
    print(f"{seg:<10} | {val1:<12.4f} | {val2:<12.4f} | {val3:<12.4f}")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Define the directories for each system
systems_paths = {
    "Natural 2B": "/project/zerze/krahimin/proteinA_all/2B/round3/data/rmsf/",
    "2B-Mutated linker": "/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/rmsf/",
    "Natural ED": "/project/zerze/krahimin/proteinA_all/ED_round3/data/rmsf/"
}

# Updated segments as requested
segments = ["all", "linker", "D1T1", "D1T2", "D2T1", "D2T2"]

# Colors mapping: Natural 2B (Black), Mutated (Red), Natural ED (Green)
colors = ['black', 'red', 'green']

def get_average_rmsf(filepath):
    """Parses a GROMACS .xvg file and returns the mean RMSF."""
    try:
        # Load data, skipping lines starting with # or @
        data = np.loadtxt(filepath, comments=['#', '@'])
        # Column 1 contains the RMSF values in nm
        return np.mean(data[:, 1])
    except (FileNotFoundError, OSError):
        return 0.0
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return 0.0

# Dictionary to store calculated averages
results = {name: [] for name in systems_paths.keys()}

# Data Processing
for system_name, base_path in systems_paths.items():
    for seg in segments:
        file_name = f"rmsf_{seg}.xvg"
        full_path = os.path.join(base_path, file_name)
        avg_val = get_average_rmsf(full_path)
        results[system_name].append(avg_val)

# Visualization
x = np.arange(len(segments))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6), dpi=100)

# Plotting each system with designated colors
for i, (system_name, averages) in enumerate(results.items()):
    offset = (i - 1) * width 
    ax.bar(x + offset, 
           averages, 
           width, 
           label=system_name, 
           color=colors[i], 
           edgecolor='black', 
           linewidth=1.0,
           alpha=0.9)

# Plot Formatting
ax.set_ylabel('Average RMSF (nm)', fontsize=20, fontweight='bold')
ax.set_title('Average RMSF: Domains and Linker Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(segments, fontsize=20)
ax.legend(frameon=False)
ax.grid(axis='y', linestyle=':', alpha=0.7)

# Adjust y-limit for better visual headroom
max_val = max([max(v) for v in results.values() if v])
ax.set_ylim(0, max_val * 1.2)

plt.tight_layout()
plt.savefig('rmsf_domain_comparison.png')
plt.show()

# Print text summary for quick review
header = f"{'Segment':<10} | {'Natural 2B':<12} | {'Mutated':<12} | {'Natural ED':<12}"
print(header)
print("-" * len(header))
for idx, seg in enumerate(segments):
    v = [results[sys][idx] for sys in systems_paths.keys()]
    print(f"{seg:<10} | {v[0]:<12.4f} | {v[1]:<12.4f} | {v[2]:<12.4f}")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Define the system configurations with their specific paths and residue ranges
# Residue ranges are defined as (start, end) inclusive.
systems_config = {
    "Natural 2B": {
        "gro": "/project/zerze/krahimin/proteinA_all/2B/round3/em.gro",
        "rmsf": "/project/zerze/krahimin/proteinA_all/2B/round3/data/rmsf/rmsf_all.xvg",
        "color": "black",
        "ranges": {
            "all": (1, 158),
            "D1H1": (8, 19), "D1T1": (20, 24), "D1H2": (25, 36), "D1T2": (37, 40), "D1H3": (41, 55),
            "linker": (56, 107),
            "D2H1": (108, 119), "D2T1": (120, 124), "D2H2": (125, 136), "D2T2": (137, 140), "D2H3": (141, 155)
        }
    },
    "2B-Mutated linker": {
        "gro": "/project/zerze/krahimin/proteinA_all/2B/round4_linker/em.gro",
        "rmsf": "/project/zerze/krahimin/proteinA_all/2B/round4_linker/data/rmsf/rmsf_all.xvg",
        "color": "red",
        "ranges": {
            "all": (1, 158),
            "D1H1": (8, 19), "D1T1": (20, 24), "D1H2": (25, 36), "D1T2": (37, 40), "D1H3": (41, 55),
            "linker": (56, 107),
            "D2H1": (108, 119), "D2T1": (120, 124), "D2H2": (125, 136), "D2T2": (137, 140), "D2H3": (141, 155)
        }
    },
    "Natural ED": {
        "gro": "/project/zerze/krahimin/proteinA_all/ED_round3/em.gro",
        "rmsf": "/project/zerze/krahimin/proteinA_all/ED_round3/data/rmsf/rmsf_all.xvg",
        "color": "green",
        "ranges": {
            "all": (1, 150), # Based on D2H3 range
            "D1H1": (42, 53), "D1T1": (54, 58), "D1H2": (59, 70), "D1T2": (71, 74), "D1H3": (75, 89),
            "linker": (90, 102),
            "D2H1": (103, 114), "D2T1": (115, 119), "D2H2": (120, 131), "D2T2": (132, 135), "D2H3": (136, 150)
        }
    }
}

def get_indices_from_gro(file_path, ranges_dict):
    """Maps residue ranges to atom indices found in the .gro file."""
    indices_map = {key: [] for key in ranges_dict}
    if not os.path.exists(file_path):
        return indices_map
    with open(file_path, 'r') as f:
        lines = f.readlines()[2:-1] # Skip header and box vectors
        for line in lines:
            try:
                res_num = int(line[0:5].strip())
                atom_num = int(line[15:20].strip())
                for key, (start, end) in ranges_dict.items():
                    if start <= res_num <= end:
                        indices_map[key].append(atom_num)
            except: continue
    return indices_map

def calculate_means(rmsf_file, indices_map, order):
    """Calculates mean RMSF for specific atom groups."""
    if not os.path.exists(rmsf_file):
        return [0.0] * len(order)
    data = np.loadtxt(rmsf_file, comments=['#', '@'])
    rmsf_lookup = dict(zip(data[:,0].astype(int), data[:,1]))
    means = []
    for segment in order:
        indices = indices_map.get(segment, [])
        vals = [rmsf_lookup[idx] for idx in indices if idx in rmsf_lookup]
        means.append(np.mean(vals) if vals else 0.0)
    return means

# Execution and Plotting
helix_order = ["D1H1", "D1H2", "D1H3", "D2H1", "D2H2", "D2H3"]
other_order = ["all", "linker", "D1T1", "D1T2", "D2T1", "D2T2"]

results = {}
for name, config in systems_config.items():
    idx_map = get_indices_from_gro(config["gro"], config["ranges"])
    results[name] = {
        "helices": calculate_means(config["rmsf"], idx_map, helix_order),
        "others": calculate_means(config["rmsf"], idx_map, other_order),
        "color": config["color"]
    }

def create_plot(order, data_key, filename):
    x = np.arange(len(order))
    width = 0.25
    fig, ax = plt.subplots(figsize=(12, 6), dpi=400)
    for i, (name, val) in enumerate(results.items()):
        ax.bar(x + (i - 1) * width, val[data_key], width, label=name, 
               color=val["color"], edgecolor='black', alpha=0.85)
    ax.set_ylabel(r'$\text{Average RMSF (nm)}$', fontsize=20, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(order, fontsize=20)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), 
              ncol=3, frameon=False, fontsize=20)
    ax.grid(axis='y', linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.savefig(filename)

# Generate the two requested plots
create_plot(helix_order, "helices", "rmsf_helices_comparison.png")
create_plot(other_order, "others", "rmsf_global_turns_comparison.png")

plt.show()